---
## 1. Install Packages

In [1]:
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

import torch
TORCH_VER = torch.__version__.split('+')[0]
CUDA_TAG  = (torch.version.cuda or '').replace('.', '')
print(f'PyTorch {TORCH_VER} | CUDA {torch.version.cuda} | GPUs: {torch.cuda.device_count()}')

PYG_URL = f'https://data.pyg.org/whl/torch-{TORCH_VER}+cu{CUDA_TAG}.html'
!pip install -q torch_geometric
!pip install -q pyg_lib torch_scatter torch_sparse torch_cluster -f {PYG_URL}
!pip install -q rdkit openpyxl tqdm scipy scikit-learn requests pyarrow transformers

import torch_geometric
print(f'torch_geometric {torch_geometric.__version__} OK')

Thu May  7 19:18:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# -- Mamba install (CUDA only, ~10 min compile) ------------------------------
# Set INSTALL_MAMBA = False to run Version A only (no Mamba encoders).
INSTALL_MAMBA = True

if INSTALL_MAMBA:
    if not torch.cuda.is_available():
        raise RuntimeError('Mamba requires a CUDA GPU. Enable a GPU accelerator in Kaggle settings.')
    print('Installing causal-conv1d ...')
    !pip install -q causal-conv1d>=1.4.0
    print('Installing mamba-ssm (compiles CUDA kernels) ...')
    !pip install -q mamba-ssm --no-build-isolation
    from mamba_ssm import Mamba as _MambaTest
    _m = _MambaTest(d_model=64, d_state=16, d_conv=4).cuda()
    _ = _m(torch.randn(2, 8, 64).cuda())
    del _m, _MambaTest
    print('mamba_ssm smoke test PASSED \u2713')
else:
    print('Mamba install skipped (INSTALL_MAMBA=False). Only Version A will run.')


Installing causal-conv1d ...
  error: subprocess-exited-with-error
  
  × Building wheel for causal-conv1d (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for causal-conv1d
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (causal-conv1d)
Installing mamba-ssm (compiles CUDA kernels) ...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.7/121.7 kB 3.5 MB/s eta 0:00:00
  Preparing metadata (pyproject.toml) ... done
mamba_ssm smoke test PASSED ✓


---
## 2. Download All Data

In [3]:
import os, json, time, requests
from pathlib import Path
from tqdm.auto import tqdm
import pandas as pd
import numpy as np

WORK = Path('/kaggle/working')
RAW  = WORK / 'data' / 'raw'
PROC = WORK / 'data' / 'processed'
for d in [RAW, PROC]: d.mkdir(parents=True, exist_ok=True)

def _download(url, dest, min_mb=0, desc=None):
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size >= min_mb * 1024**2:
        print(f'  {dest.name}: already present ({dest.stat().st_size/1024**2:.0f} MB)  -  skip')
        return
    print(f'  Downloading {desc or dest.name} ...')
    hdrs = {'User-Agent': 'pathxdrp-kaggle/1.0'}
    r = requests.get(url, stream=True, timeout=600, headers=hdrs)
    r.raise_for_status()
    total = int(r.headers.get('content-length', 0))
    tmp = dest.with_suffix('.tmp')
    with open(tmp, 'wb') as f, tqdm(total=total or None, unit='B', unit_scale=True, desc=desc or dest.name) as pb:
        for chunk in r.iter_content(1 << 20):
            f.write(chunk); pb.update(len(chunk))
    tmp.rename(dest)
    print(f'  Saved {dest.name}: {dest.stat().st_size/1024**2:.0f} MB')

# -- GDSC2 IC50 data (from Sanger Institute) -------------------------------------
GDSC2_URL = 'https://cog.sanger.ac.uk/cancerrxgene/GDSC_release8.5/GDSC2_fitted_dose_response_27Oct23.xlsx'
GDSC2_XLS = RAW / 'GDSC2_fitted_dose_response.xlsx'
_download(GDSC2_URL, GDSC2_XLS, min_mb=10, desc='GDSC2 IC50')

# -- Screened compounds / drug metadata (Sanger) -------------------------
CPD_URL = 'https://cog.sanger.ac.uk/cancerrxgene/GDSC_release8.5/screened_compounds_rel_8.5.csv'
CPD_CSV = RAW / 'screened_compounds.csv'
_download(CPD_URL, CPD_CSV, min_mb=0.01, desc='screened compounds')

# -- Cell Lines Details (Sanger, for tissue labels) ----------------------
CELL_URL = 'https://cog.sanger.ac.uk/cancerrxgene/GDSC_release8.5/Cell_Lines_Details.xlsx'
CELL_XLS = RAW / 'Cell_Lines_Details.xlsx'
_download(CELL_URL, CELL_XLS, min_mb=0.05, desc='Cell Lines Details')

# -- DepMap Model.csv (COSMIC->ACH mapping) ------------------------------------------
# Use DepMap portal API with explicit release so the URL is stable.
MODEL_CSV = PROC / 'Model.csv'
if not MODEL_CSV.exists():
    _model_url = ('https://depmap.org/portal/download/api/download'
                  '?file_name=downloads-by-canonical-id%2Fpublic-26q1-5bbf.37%2FModel.csv'
                  '&dl_name=Model.csv&bucket=depmap-external-downloads')
    try:
        _download(_model_url, MODEL_CSV, desc='DepMap Model.csv')
    except Exception as _e:
        print(f'  WARNING: Model.csv download failed ({_e}). Will use SANGER_MODEL_ID fallback.')
else:
    print(f'  {MODEL_CSV.name}: already present')

# -- DepMap expression matrix (~305 MB, DepMap portal 26Q1) -------------------------
# 26Q1 renamed the file and made rows ProfileID-indexed (ModelID is a column).
EXPR_FNAME = 'OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv'
EXPR_FILE  = RAW / EXPR_FNAME
_expr_url  = ('https://depmap.org/portal/download/api/download'
              '?file_name=downloads-by-canonical-id%2Fpublic-26q1-5bbf.27%2F'
              'OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv'
              '&dl_name=OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv'
              '&bucket=depmap-external-downloads')
_download(_expr_url, EXPR_FILE, min_mb=250, desc='DepMap expression (~305 MB)')

print('\nAll downloads complete.')


GDSC2 IC50:   0%|          | 0.00/21.3M [00:00<?, ?B/s]

  Saved GDSC2_fitted_dose_response.xlsx: 20 MB


screened compounds:   0%|          | 0.00/46.4k [00:00<?, ?B/s]

  Saved screened_compounds.csv: 0 MB


Cell Lines Details:   0%|          | 0.00/117k [00:00<?, ?B/s]

  Saved Cell_Lines_Details.xlsx: 0 MB


DepMap Model.csv:   0%|          | 0.00/697k [00:00<?, ?B/s]

  Saved Model.csv: 1 MB


DepMap expression (~305 MB):   0%|          | 0.00/305M [00:00<?, ?B/s]

  Saved OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv: 291 MB

All downloads complete.


---
## 3. Preprocessing

In [4]:
# -- 3.1  Build GDSC2 dataset CSV and compounds annotation -------------------

GDSC2_CSV = PROC / 'GDSC2-dataset.csv'
DRUGS_CSV = PROC / 'Compounds-annotation.csv'

if not GDSC2_CSV.exists():
    print('Reading GDSC2 xlsx (may take ~30 s) ...')
    gdsc2 = pd.read_excel(GDSC2_XLS, engine='openpyxl')
    gdsc2.to_csv(GDSC2_CSV, index=False)
    print(f'  Saved GDSC2 CSV: {len(gdsc2):,} rows')
else:
    gdsc2 = pd.read_csv(GDSC2_CSV)
    print(f'GDSC2 CSV: {len(gdsc2):,} rows (cached)')

# Validate that the IC50 target column exists
if 'LN_IC50' not in gdsc2.columns:
    raise KeyError(
        f'Expected column "LN_IC50" not found in GDSC2 data.\n'
        f'Available columns: {list(gdsc2.columns)}\n'
        'Check the downloaded GDSC2 release — column names may differ between releases.')
print(f'GDSC2 columns: {list(gdsc2.columns)}')

def _build_drugs_csv(cpd_path, gdsc2_df):
    cpd = pd.read_csv(cpd_path)
    # Normalise columns — Sanger CSV uses different casing/spacing in different releases
    cpd.columns = [c.strip().upper().replace(' ', '_') for c in cpd.columns]
    print(f'  screened_compounds columns: {list(cpd.columns)}')
    rename = {
        'DRUG_ID':       'DRUG_ID',
        'DRUG_NAME':     'DRUG_NAME',
        'SYNONYMS':      'SYNONYMS',
        'TARGET':        'TARGET',
        'TARGET_PATHWAY': 'TARGET_PATHWAY',
        'PUTATIVE_TARGET': 'TARGET',
        'PATHWAY_NAME':  'TARGET_PATHWAY',
        'SMILES':        'SMILES',    # preserve if present
    }
    cpd = cpd.rename(columns={k: v for k, v in rename.items() if k in cpd.columns})
    # Fall back to GDSC2 response for drug metadata if columns are missing
    for col in ['DRUG_ID', 'DRUG_NAME']:
        if col not in cpd.columns:
            print(f'  WARNING: {col} missing from screened_compounds; deriving from GDSC2 response.')
            cpd = gdsc2_df[['DRUG_ID', 'DRUG_NAME', 'PUTATIVE_TARGET', 'PATHWAY_NAME']
                           ].drop_duplicates('DRUG_ID').copy()
            cpd = cpd.rename(columns={'PUTATIVE_TARGET': 'TARGET', 'PATHWAY_NAME': 'TARGET_PATHWAY'})
            break
    for col in ['SYNONYMS', 'TARGET', 'TARGET_PATHWAY']:
        if col not in cpd.columns:
            cpd[col] = None
    # Save — include SMILES column if it was in the source file
    keep = ['DRUG_ID', 'DRUG_NAME', 'SYNONYMS', 'TARGET', 'TARGET_PATHWAY']
    if 'SMILES' in cpd.columns:
        keep.append('SMILES')
        n_smi = cpd['SMILES'].notna().sum()
        print(f'  SMILES found in screened_compounds: {n_smi}/{len(cpd)}')
    else:
        print('  No SMILES column in screened_compounds — will fetch from PubChem in step 3.3.')
    cpd[keep].to_csv(DRUGS_CSV, index=False)
    print(f'  Compounds annotation saved: {len(cpd)} drugs, columns: {keep}')

if not DRUGS_CSV.exists():
    _build_drugs_csv(CPD_CSV, gdsc2)
else:
    _cpd_existing = pd.read_csv(DRUGS_CSV)
    print(f'Compounds annotation: {len(_cpd_existing)} drugs (cached), '
          f'columns: {list(_cpd_existing.columns)}')
    # If the cached file has no SMILES column, rebuild to pick it up from source if now available
    if 'SMILES' not in _cpd_existing.columns:
        print('  Cached file has no SMILES column — checking source for SMILES ...')
        _tmp = pd.read_csv(CPD_CSV)
        _tmp.columns = [c.strip().upper().replace(' ', '_') for c in _tmp.columns]
        if 'SMILES' in _tmp.columns and _tmp['SMILES'].notna().any():
            print('  Source HAS SMILES — rebuilding Compounds-annotation.csv ...')
            DRUGS_CSV.unlink()
            _build_drugs_csv(CPD_CSV, gdsc2)
        else:
            print('  Source has no SMILES either — will rely on PubChem fetch in step 3.3.')


Reading GDSC2 xlsx (may take ~30 s) ...
  Saved GDSC2 CSV: 242,036 rows
GDSC2 columns: ['DATASET', 'NLME_RESULT_ID', 'NLME_CURVE_ID', 'COSMIC_ID', 'CELL_LINE_NAME', 'SANGER_MODEL_ID', 'TCGA_DESC', 'DRUG_ID', 'DRUG_NAME', 'PUTATIVE_TARGET', 'PATHWAY_NAME', 'COMPANY_ID', 'WEBRELEASE', 'MIN_CONC', 'MAX_CONC', 'LN_IC50', 'AUC', 'RMSE', 'Z_SCORE']
  screened_compounds columns: ['DRUG_ID', 'SCREENING_SITE', 'DRUG_NAME', 'SYNONYMS', 'TARGET', 'TARGET_PATHWAY']
  No SMILES column in screened_compounds — will fetch from PubChem in step 3.3.
  Compounds annotation saved: 621 drugs, columns: ['DRUG_ID', 'DRUG_NAME', 'SYNONYMS', 'TARGET', 'TARGET_PATHWAY']


In [5]:
# -- 3.2  Build COSMIC->DepMap ModelID mapping --------------------------------

COSMIC_MAP = PROC / 'cosmic_to_depmap.csv'

def _build_from_model_csv(path):
    model_df = pd.read_csv(path)
    model_df.columns = [c.strip() for c in model_df.columns]
    print(f'  Model.csv columns: {list(model_df.columns)}')
    cosmic_col = next((c for c in model_df.columns if 'COSMIC' in c.upper()), None)
    model_col  = next((c for c in model_df.columns
                       if c in ('ModelID', 'DepMap_ID', 'model_id', 'depmap_id')), None)
    if model_col is None:
        model_col = next((c for c in model_df.columns
                          if 'model' in c.lower() and 'id' in c.lower()), None)
    if not cosmic_col:
        raise ValueError(
            f'No COSMIC ID column found in Model.csv. Available: {list(model_df.columns)}\n'
            'Expected a column whose name contains "COSMIC".')
    if not model_col:
        raise ValueError(
            f'No ModelID column found in Model.csv. Available: {list(model_df.columns)}\n'
            'Expected one of: ModelID, DepMap_ID, model_id, depmap_id.')
    mp = model_df[[model_col, cosmic_col]].copy()
    mp.columns = ['ModelID', 'COSMICID']
    mp['COSMICID'] = pd.to_numeric(mp['COSMICID'], errors='coerce')
    mp = mp.dropna(subset=['COSMICID'])
    if len(mp) == 0:
        raise ValueError('Model.csv: no valid (ModelID, COSMICID) pairs after parsing.')
    mp['COSMICID'] = mp['COSMICID'].astype(int)
    mp.to_csv(COSMIC_MAP, index=False)
    print(f'  cosmic_to_depmap.csv built from Model.csv: {len(mp)} entries')

if not COSMIC_MAP.exists():
    if not MODEL_CSV.exists():
        raise FileNotFoundError(
            f'Model.csv not found at {MODEL_CSV}.\n'
            'Download it from https://depmap.org/portal/download/all/ '
            'and save to that path before continuing.')
    _build_from_model_csv(MODEL_CSV)
else:
    print(f'cosmic_to_depmap.csv: {len(pd.read_csv(COSMIC_MAP))} entries (cached)')


  Model.csv columns: ['ModelID', 'PatientID', 'CellLineName', 'StrippedCellLineName', 'DepmapModelType', 'OncotreeLineage', 'OncotreePrimaryDisease', 'OncotreeSubtype', 'OncotreeCode', 'PatientSubtypeFeatures', 'RRID', 'Age', 'AgeCategory', 'Sex', 'PatientRace', 'PrimaryOrMetastasis', 'SampleCollectionSite', 'SourceType', 'SourceDetail', 'CatalogNumber', 'ModelType', 'TissueOrigin', 'ModelDerivationMaterial', 'ModelTreatment', 'PatientTreatmentStatus', 'PatientTreatmentType', 'PatientTreatmentDetails', 'Stage', 'StagingSystem', 'PatientTumorGrade', 'PatientTreatmentResponse', 'GrowthPattern', 'OnboardedMedia', 'FormulationID', 'SerumFreeMedia', 'PlateCoating', 'EngineeredModel', 'EngineeredModelDetails', 'CulturedResistanceDrug', 'PublicComments', 'CCLEName', 'HCMIID', 'PediatricModelType', 'ModelAvailableInDbgap', 'ModelSubtypeFeatures', 'WTSIMasterCellID', 'SangerModelID', 'COSMICID', 'ModelIDAlias']
  cosmic_to_depmap.csv built from Model.csv: 977 entries


In [6]:
# -- 3.3  Build SMILES table ------------------------------------------------
# _SMILES_SEED: 502 DRUG_ID -> SMILES pairs pre-fetched from PubChem locally.
# For any drugs not in this dict, fall back to PubChem then ChEMBL (needs internet).
_SMILES_SEED = {
    1: "COCCOC1=C(C=C2C(=C1)C(=NC=N2)NC3=CC=CC(=C3)C#C)OCCOC",
    3: "C[C@@H]1CC[C@H]2C[C@@H](/C(=C/C=C/C=C/[C@H](C[C@H](C(=O)[C@@H]([C@@H](/C(=C/[C@H](C(=O)C[C@H](OC(=O)[C@@H]3CCCCN3C(=O)C(=O)[C@@]1(O2)O)[C@H](C)C[C@@H]4CC[C@H]([C@@H](C4)OC)O)C)/C)O)OC)C)C)/C)OC",
    5: "CCN(CC)CCNC(=O)C1=C(NC(=C1C)/C=C\\2/C3=C(C=CC(=C3)F)NC2=O)C",
    6: "CC1=C(NC(=C1C(=O)N2CCC[C@@H]2CN3CCCC3)C)/C=C\\4/C5=C(C=CC(=C5)S(=O)(=O)CC6=C(C=CC=C6Cl)Cl)NC4=O",
    9: "CC(C)C[C@@H](C=O)NC(=O)[C@H](CC(C)C)NC(=O)[C@H](CC(C)C)NC(=O)OCC1=CC=CC=C1",
    11: "CC1=C2[C@H](C(=O)[C@@]3([C@H](C[C@@H]4[C@]([C@H]3[C@@H]([C@@](C2(C)C)(C[C@@H]1OC(=O)[C@@H]([C@H](C5=CC=CC=C5)NC(=O)C6=CC=CC=C6)O)O)OC(=O)C7=CC=CC=C7)(CO4)OC(=O)C)O)C)OC(=O)C",
    17: "C[C@H]1C[C@@H]2[C@H]([C@H]([C@]3(O2)CC[C@H]4[C@@H]5CC=C6C[C@H](CC[C@@]6([C@H]5CC4=C3C)C)O)C)NC1",
    29: "CC1=C(C=C(C=C1)NC(=O)C2=CC(=CC=C2)C(C)(C)C#N)NC3=CC4=C(C=C3)N=CN(C4=O)C",
    30: "CNC(=O)C1=NC=CC(=C1)OC2=CC=C(C=C2)NC(=O)NC3=CC(=C(C=C3)Cl)C(F)(F)F",
    32: "CC1=CC(=NN1)NC2=CC(=NC(=N2)SC3=CC=C(C=C3)NC(=O)C4CC4)N5CCN(CC5)C",
    34: "CC1=C(C=C(C=C1)NC(=O)C2=CC=C(C=C2)CN3CCN(CC3)C)NC4=NC=CC(=N4)C5=CN=CC=C5",
    35: "CC(C)S(=O)(=O)C1=CC=CC=C1NC2=NC(=NC=C2Cl)NC3=C(C=C(C=C3)N4CCC(CC4)N5CCN(CC5)C)OC",
    37: "C[C@H](C1=C(C=CC(=C1Cl)F)Cl)OC2=C(N=CC(=C2)C3=CN(N=C3)C4CCNCC4)N",
    38: "CN1CCN(CC1)CCOC2=CC3=C(C(=C2)OC4CCOCC4)C(=NC=N3)NC5=C(C=CC6=C5OCO6)Cl",
    41: "C1=CC=C(C=C1)C(C2=CC=CC=C2)(C3=CC=CC=C3)SC[C@@H](C(=O)O)N",
    45: "CCCC[C@@H](C=O)NC(=O)[C@H](CC(C)C)NC(=O)[C@H](CC(C)C)NC(=O)OCC1=CC=CC=C1",
    51: "CC1=C(C(=CC=C1)Cl)NC(=O)C2=CN=C(S2)NC3=CC(=NC(=N3)C)N4CCN(CC4)CCO",
    52: "C1=CC(=CC(=C1)C(=O)N)C2=CC(=NC=N2)NC3=CC=C(C=C3)OC(F)(F)F",
    53: "C1=CC(=CC(=C1)Cl)NC2=NC=CC(=N2)C3=CC(=NC=C3)NCCCO",
    54: "CCNC1=CC(=NC(=N1)NC2=CC3=C(C=C2)N(C=C3)CC4=CC=CC=C4)NC5CCC(CC5)O",
    55: "CC(=O)N1CCN(CC1)C2CCC(CC2)N3C4=NC=NC(=C4C(=N3)C5=CC(=C(C=C5)NC(=O)C6=CC7=CC=CC=C7N6C)OC)N",
    56: "CC1=C(C(=CC=C1)C)OC(=O)N(C2=C(C=C(C=C2)OC)OC)C3=NC(=NC=C3)NC4=CC=C(C=C4)N5CCN(CC5)C",
    59: "CCC(=O)N(C)C1=CC=C(C=C1)NC2=NC3=C(C(=N2)NC4CCCN(C4)C(=O)C=C)NC=N3",
    60: "CC[C@@H]1C(=O)N(C2=CN=C(N=C2N1C3CCCC3)NC4=C(C=C(C=C4)C(=O)NC5CCN(CC5)C)OC)C",
    62: "CC1=CC(=CC2=C1N=C(N2)C3=C(C=CNC3=O)NC[C@H](C4=CC(=CC=C4)Cl)O)N5CCOCC5",
    63: "CC1=CC(=C(C=C1SC2=CN=C(S2)NC(=O)C3=CC=C(C=C3)CNC(C)C(C)(C)C)C(=O)N4CCN(CC4)C(=O)C)OC",
    64: "CC1=CC=C(C=C1)C2=C(N(C3=NC=NC(=C23)N)CCCO)C(=O)CCl",
    71: "CCC1=C(C(=NC(=N1)N)N)C2=CC=C(C=C2)Cl",
    86: "CC1=C2C=C(C=CC2=NN1)C3=CC(=CN=C3)OC[C@H](CC4=CNC5=CC=CC=C54)N",
    87: "COC1=C(C=C2C(=C1)N=CN2C3=CC(=C(S3)C(=O)N)OCC4=CC=CC=C4C(F)(F)F)OC",
    88: "C1=CC=C(C(=C1)N)NC(=O)C2=CC=C(C=C2)CNC(=O)OCC3=CN=CC=C3",
    89: "C/C/1=C\\CC[C@@]2([C@@H](O2)[C@@H]3[C@@H](CC1)C(=C)C(=O)O3)C",
    91: "COC1=C(C=C2C(=C1)N=CN2C3=CC(=C(S3)C#N)OCC4=CC=CC=C4S(=O)(=O)C)OC",
    94: "CC1=CN2C(=O)C=C(N=C2C(=C1)C(C)NC3=CC=CC=C3)N4CCOCC4",
    104: "B([C@H](CC(C)C)NC(=O)[C@H](CC1=CC=CC=C1)NC(=O)C2=NC=CN=C2)(O)O",
    106: "CN1CCN(CC1)C2=CC(=C(C=C2)NC3=NC=C4C(=N3)N(C5=CC=CC=C5C(=O)N4C)C)OC",
    110: "CC[C@H](CO)NC1=NC(=C2C(=N1)N(C=N2)C(C)C)NCC3=CC=CC=C3",
    111: "C1=CC=C(C=C1)/C=C/C(=O)NC(C(Cl)(Cl)Cl)NC(=S)NC2=CC=CC3=C2N=CC=C3",
    119: "CS(=O)(=O)CCNCC1=CC=C(O1)C2=CC3=C(C=C2)N=CN=C3NC4=CC(=C(C=C4)OCC5=CC(=CC=C5)F)Cl",
    127: "CCN1C2=CC(=NC=C2N=C1C3=NON=C3N)OC4=CC=CC(=C4)NC(=O)C5=CC=C(C=C5)OCCN6CCOCC6",
    133: "C[C@H]1[C@H]([C@H](C[C@@H](O1)O[C@H]2C[C@@](CC3=C2C(=C4C(=C3O)C(=O)C5=C(C4=O)C(=CC=C5)OC)O)(C(=O)CO)O)N)O",
    134: "C[C@@H]1OC[C@@H]2[C@@H](O1)[C@@H]([C@H]([C@@H](O2)O[C@H]3[C@H]4COC(=O)[C@@H]4[C@@H](C5=CC6=C(C=C35)OCO6)C7=CC(=C(C(=C7)OC)O)OC)O)O",
    135: "C1=CN(C(=O)N=C1N)[C@H]2C([C@@H]([C@H](O2)CO)O)(F)F",
    136: "CC1=C(C(=O)C2=C(C1=O)N3C[C@H]4[C@@H]([C@@]3([C@@H]2COC(=O)N)OC)N4)N",
    140: "CCC1=C[C@H]2C[C@@](C3=C(CN(C2)C1)C4=CC=CC=C4N3)(C5=C(C=C6C(=C5)[C@]78CCN9[C@H]7[C@@](C=CC9)([C@H]([C@@]([C@@H]8N6C)(C(=O)OC)O)OC(=O)C)CC)OC)C(=O)OC",
    147: "C1=CC2=C(C=C(C(=C2N=C1)O)N=NC3=CC4=C(C=C3)C=C(C=C4)S(=O)(=O)O)S(=O)(=O)O",
    150: "CC(CS(=O)(=O)C1=CC=C(C=C1)F)(C(=O)NC2=CC(=C(C=C2)C#N)C(F)(F)F)O",
    151: "C1CC2=C(C1)C=C(C=C2)OC3=NC(=C4C(=N3)N(C=N4)CC5=CC=C(C=C5)C6=CC=CC=C6)N[C@@H](CC7=CC=CC=C7)CO",
    152: "COC1=C(C=C2C(=C1)C(=NC=N2)N3C(=NC(=N3)C4=CC=CC=N4)N)OC",
    153: "C[C@@]12[C@@H]([C@@H](C[C@@H](O1)N3C4=CC=CC=C4C5=C6C(=C7C8=CC=CC=C8N2C7=C53)CNC6=O)N(C)C(=O)C9=CC=CC=C9)OC",
    154: "CC1=CN=C(N1)C2=CN=C(N=C2C3=C(C=C(C=C3)Cl)Cl)NCCNC4=NC=C(C=C4)C#N",
    155: "CC1=C(C=C(C=C1)C(=O)NC2=CC(=C(C=C2)CN3CCN(CC3)C)C(F)(F)F)C#CC4=CN=C5N4N=CC=C5",
    156: "CC1=CN2C(=O)C=C(N=C2C(=C1)[C@@H](C)NC3=CC=CC=C3C(=O)O)N4CCOCC4",
    158: "CN(C1=C(C=CC=N1)CNC2=NC(=NC=C2C(F)(F)F)NC3=CC4=C(C=C3)NC(=O)C4)S(=O)(=O)C",
    159: "CCN1CCN(CC1)CC2=C(C=C(C=C2)NC(=O)C3=CC(=C(C=C3)C)/C=C/C4=CN=C5C(=C4OC)C=CN5)C(F)(F)F",
    163: "CC1=C(SC2=C1C(=N[C@H](C3=NN=C(N32)C)CC(=O)OC(C)(C)C)C4=CC=C(C=C4)Cl)C",
    165: "COC(=O)CNC(=O)C(=O)OC",
    166: "COC(=O)[C@H](CCSC)NC(=O)C1=C(C=C(C=C1)NC[C@H](CS)N)C2=CC=CC=C2",
    167: "C1=CC=C2C(=C1)C=CC3=C2C=CC(=C3)C4=CC(=NN4C5=CC=C(C=C5)NC(=O)CN)C(F)(F)F",
    170: "CC(=CC[C@H](C1=CC(=O)C2=C(C=CC(=C2C1=O)O)O)O)C",
    171: "C1CN(CCC1N2C3=CC=CC=C3NC2=O)CC4=CC=C(C=C4)C5=C(N=C6C=C7C(=NC=N7)C=C6N5)C8=CC=CC=C8",
    172: "CCCCCCCCCCCC1=C(C(=O)C=C(C1=O)O)O",
    173: "CC1=C(C=CC(=C1)[N+](=O)[O-])NS(=O)(=O)C2=C(C=CC(=C2)Cl)Cl",
    175: "C=CCC1=C(C(=CC=C1)/C=N/NC(=O)CN2CCN(CC2)CC3=CC=CC=C3)O",
    176: "C1=CC=C2C(=C1)C=CC(=C2SSC3=C(C=CC4=CC=CC=C43)O)O",
    177: "C1CCC(C1)C2=C(C=CC(=C2)C3=CNC4=C3C=C(C=N4)C5=CC=CC=C5)C(=O)O",
    178: "COC1=C(C=C(C=C1)C2=CC3=NC=CN3C(=N2)NC4=C(C=CC=N4)C(=O)N)OC.Cl.Cl",
    179: "C1=C(C(=O)NC(=O)N1)F",
    180: "CCCCCCCC(=O)O[C@H]1[C@H]2C(=C([C@@H]1OC(=O)/C(=C\\C)/C)C)[C@H]3[C@]([C@H](C[C@]2(C)OC(=O)C)OC(=O)CCC)([C@](C(=O)O3)(C)O)O",
    182: "CC1=CC(=C(N1)/C=C\\2/C(=C/C(=C/3\\C=C4C=CC=CC4=N3)/N2)OC)C.CS(=O)(=O)O",
    184: "C[C@]1(CCCN1C2=NN3C=CC=C3C(=N2)NC4=NNC(=C4)C5CC5)C(=O)NC6=CN=C(C=C6)F",
    185: "CC1(CC(C1)C2=NC(=C3N2C=CN=C3N)C4=CC5=C(C=C4)C=CC(=N5)C6=CC=CC=C6)O",
    186: "CC1=CC2=C(C=C1C(=C)C3=CC=C(C=C3)C(=O)O)C(CCC2(C)C)(C)C",
    190: "CC1=C(N=C(N=C1N)[C@H](CC(=O)N)NC[C@@H](C(=O)N)N)C(=O)N[C@@H]([C@H](C2=CN=CN2)OC3C(C(C(C(O3)CO)O)O)OC4C(C(C(C(O4)CO)O)OC(=O)N)O)C(=O)N[C@H](C)[C@H]([C@H](C)C(=O)N[C@@H]([C@@H](C)O)C(=O)NCCC5=NC(=CS5)C6=NC(=CS6)C(=O)NCCC[S+](C)C)O",
    192: "C/C(=C(\\C#N)/C(=O)NC1=C(C=CC(=C1)Br)Br)/O",
    193: "COC1=CC=C(C=C1)COC2=C(C=C(C=C2)CC3=CN=C(N=C3N)N)OC",
    194: "CCNC(=O)C1=NOC(=C1C2=CC=C(C=C2)CN3CCOCC3)C4=CC(=C(C=C4O)O)C(C)C",
    196: "C1=CC=C(C=C1)CCN=C(N)N=C(N)N",
    197: "CCC/C=C/C=C/C(=O)O[C@H]1/C(=C/C(=O)OC)/C[C@H]2C[C@@H](OC(=O)C[C@@H](C[C@@H]3C[C@@H](C([C@@](O3)(C[C@@H]4C/C(=C/C(=O)OC)/C[C@@H](O4)/C=C/C([C@@]1(O2)O)(C)C)O)(C)C)OC(=O)C)O)[C@@H](C)O",
    199: "CC1=C(C=C(C=C1)NC2=NC=CC(=N2)N(C)C3=CC4=NN(C(=C4C=C3)C)C)S(=O)(=O)N",
    200: "C1=CC=C2C(=C1)C(=CN2)CCN(CCO)CC3=CC=C(C=C3)/C=C/C(=O)NO",
    201: "C[C@H]1CCC[C@@]2([C@@H](O2)C[C@H](OC(=O)C[C@@H](C(C(=O)[C@@H]([C@H]1O)C)(C)C)O)/C(=C/C3=CSC(=N3)C)/C)C",
    202: "CCC1=CC(=C(C=C1N2CCC(CC2)N3CCN(CC3)S(=O)(=O)C)OC)NC4=NC=CC(=N4)C5=C(N=C6N5C=CC=C6)C7=CC(=C(C=C7)OC)C(=O)NC8=C(C=CC=C8F)F",
    203: "CC1=CC2=C(C=C1)N=C(C3=NC=C(N23)C)NCCN.Cl",
    204: "CN1C=NC=C1[C@@](C2=CC=C(C=C2)Cl)(C3=CC4=C(C=C3)N(C(=O)C=C4C5=CC(=CC=C5)Cl)C)N",
    205: "C1=CC(=CC=C1S(=O)(=O)N(CC2=C(C=C(C=C2)C3=NOC=N3)F)[C@H](CCC(F)(F)F)C(=O)N)Cl",
    206: "C1CCC(C1)[C@@H](CC#N)N2C=C(C=N2)C3=C4C=CNC4=NC=N3",
    207: "C1=CC=C2C(=C1)N=C(S2)C(C#N)C3=NC(=NC=C3)NCCC4=CN=CC=C4",
    208: "CC1=CC=C(C=C1)C(=O)N(CCCN)[C@@H](C2=NC3=C(C=CC(=C3)Cl)C(=O)N2CC4=CC=CC=C4)C(C)C.CS(=O)(=O)O",
    219: "C1CNCCC1NC(=O)C2=C(C=NN2)NC(=O)C3=C(C=CC=C3Cl)Cl",
    221: "CCC1=NC(=C(S1)C2=CC(=NC=C2)NC(=O)C3=CC=CC=C3)C4=CC=CC(=C4)C",
    222: "C1CCN(C1)C(=O)NC2=CC=CC(=C2)NC3=NC=C(C(=N3)NCCC4=CN=CN4)Br",
    223: "C1COCCN1C2=NC(=NC(=N2)N3C4=CC=CC=C4N=C3C(F)F)N5CCOCC5",
    224: "C1=CC2=NC=CN=C2C=C1/C=C/3\\C(=O)NC(=O)S3",
    226: "CCN1C=C(C(=N1)C2=CC=C(C=C2)NC(=O)N(C)C)C3=C4C=C(NC4=NC=C3)C5=CC=CC(=C5)CN(C)C",
    228: "C1CN(CCC1N2C3=CC=CC=C3NC2=O)CC4=CC=C(C=C4)C5=C(N=C6C=C7C(=NC=N7)C=C6N5)C8=CC=CC=C8",
    229: "CN1C=C(C2=CC=CC=C21)C3=C(C(=O)NC3=O)C4=CN(C5=CC=CC=C54)C6CCN(CC6)CC7=CC=CC=N7",
    230: "CC1=C(C(CC(=O)N1)C2=CC=C(C=C2)C(F)(F)F)C(=O)NC3=C(C=C4C(=C3)C=NN4)F",
    231: "CC1=CC=C(C=C1)C2=C(N(C3=NC=NC(=C23)N)CCCO)C(=O)CF",
    236: "CC1=C2C(=CC=C1)N=C(N(C2=O)C3=CC=CC=C3C)CN4C=NC5=C(N=CN=C54)N",
    238: "CC[C@@H](C1=NC2=C(C(=CC=C2)F)C(=O)N1C3=CC=CC=C3)NC4=NC=NC5=C4NC=N5",
    245: "CC(C)N1CCC(CC1)NC2=NC(=NC3=CC(=C(C=C32)OC)OCCCN4CCCC4)C5CCCCC5",
    249: "COC1=CC2=C(C=CN=C2C=C1OC)OC3=CC=C(C=C3)NC(=O)C4(CC4)C(=O)NC5=CC=C(C=C5)F",
    252: "CC(C)N1C=NC2=C(N=C(N=C21)NC3CCC(CC3)N(C)C)NC4=CC(=CC=C4)NC(=O)C=C",
    253: "CN1CCN(CC1)C2=CC(=C(C=C2)NC3=NC(=C(S3)C(=O)C4=C(C=CC=C4Cl)Cl)N)OC",
    254: "CC(C)(C)C1=CC(=NO1)NC(=O)NC2=CC=C(C=C2)C3=CN4C5=C(C=C(C=C5)OCCN6CCOCC6)SC4=N3",
    255: "CC1=NC=C(C=C1)OC2=C(C=C(C=C2)NC3=NC=NC4=C3C=C(C=C4)/C=C/CNC(=O)COC)C",
    256: "COC1=CC(=CC(=C1)C2=CC3=C4C(=CN=C3C=C2)C=CC(=O)N4C5=CC(=C(C=C5)N6CCNCC6)C(F)(F)F)OC",
    257: "CC1CC(=O)N(C2=CN=C(N=C2N1C3CCCC3)NC4=C(C=C(C=C4)C(=O)NC5CCN(CC5)C)OC)C",
    258: "CC1=CC(=CC=C1)NC2=NC(=CS2)C3=CC=NC=C3",
    260: "CCN1CCN(CC1)CC2=C(C=C(C=C2)NC(=O)C3=CC(=C(C=C3)C)OC4=C5C=CNC5=NC=C4)C(F)(F)F",
    262: "CC1=CN=C(N=C1C2=CNC(=C2)C(=O)N[C@H](CO)C3=CC(=CC=C3)Cl)NC4=C(C=C(C=C4)F)Cl",
    263: "C1=CC=C(C=C1)C2=NN3C=CC=CC3=C2C4=CC5=C(NN=C5N=N4)N",
    264: "C1=CC=C(C=C1)N(C2=CC=CC=C2)C3=NC=C(C=N3)C(=O)NCCCCCCC(=O)NO",
    265: "CN1CCC2=C(C1)C3=CC=CC=C3N2CC4=CC=C(C=C4)C(=O)NO",
    266: "CC1=CN=C(C(=N1)OC)NS(=O)(=O)C2=C(N=CC=C2)C3=CC=C(C=C3)C4=NN=CO4",
    268: "CC1=[N+](C2=C(N1CCOC)C(=O)C3=CC=CC=C3C2=O)CC4=NC=CN=C4.[Br-]",
    269: "CN1CCN(CC1)C2=CC=C(C3=NO[N+](=C23)[O-])[N+](=O)[O-]",
    272: "CC(C)[C@@H](C1=CC=CC=C1)C(=O)NC2=CC=C(C=C2)C(=O)NO",
    273: "COC1=C(C=C2C(=C1)N=CN=C2NC3=CC=CC(=C3)C#C)OCCCCCCC(=O)NO",
    274: "C1=CC=C(C=C1)NS(=O)(=O)C2=CC=CC(=C2)/C=C/C(=O)NO",
    275: "CCNC(=O)C[C@H]1C2=NN=C(N2C3=C(C=C(C=C3)OC)C(=N1)C4=CC=C(C=C4)Cl)C",
    276: "CC(C)(C)OC(=O)NC1=CC=C(C=C1)C2=CC(=NO2)C(=O)NCCCCCCC(=O)NO",
    277: "CC1=CC(=C(C=C1)F)NC(=O)NC2=CC=C(C=C2)C3=C4C(=CC=C3)NN=C4N",
    279: "CN(C)CC1=CC(=CC=C1)N=C(C2=CC=CC=C2)C3=C(NC4=C3C=CC(=C4)C(=O)N(C)C)O",
    281: "CCC1=CC2=C(C=C1N3CCC(CC3)N4CCOCC4)C(C5=C(C2=O)C6=C(N5)C=C(C=C6)C#N)(C)C",
    282: "CCOC1=C(C=C2C(=C1)N=CC(=C2NC3=CC(=C(C=C3)F)Cl)C#N)NC(=O)/C=C/CN(C)C",
    283: "COC1=C(C=C(C=N1)C2=CC3=C(C=CN=C3C=C2)C4=CN=NC=C4)NS(=O)(=O)C5=C(C=C(C=C5)F)F",
    285: "CC1=C(NC(=C1C(=O)N2CCN(CC2)C)C)/C=C\\3/C4=C(C=CC(=C4)S(=O)(=O)N(C)C5=CC(=CC=C5)Cl)NC3=O",
    287: "C1=CC=C(C=C1)C(COC2=CC3=C(C=C2)NC(=O)N3)NC(=O)C4=CC=CN(C4=O)CC5=CC(=C(C=C5)F)F",
    288: "COC1=C(C=C2C(=C1)C(=NC=N2)NC3=CC(=C(C(=C3)Br)O)Br)OC",
    290: "C1CC1COC2=CC=CC(=C2C3=NC(=C(C(=C3)C4CCNCC4)C#N)N)O",
    291: "CCN1CCC(CC1)N2C=C(N=N2)CNC3=CC4=C(C(=CN=C4C(=C3)Cl)C#N)NC5=CC(=C(C=C5)F)Cl",
    292: "CC1=C(C=C(C=C1)NC(=O)C2=CC=C(C=C2)CN3CCN(CC3)C)NC4=NC(=CS4)C5=CN=CC=C5",
    293: "C1CN(CCN1C2=NC=NC3=C2OC4=CC=CC=C43)C(=S)NCC5=CC6=C(C=C5)OCO6",
    295: "CC1=C(C=C(C=C1)C(=O)NC2=CC=CC(=C2)C(F)(F)F)NC3=C4C=NN(C4=NC(=N3)C5=CN=CC=C5)C",
    298: "C1=CC=C2C(=C1)C(=CC=N2)CNC3=C(SC=C3)C(=O)NC4=CC=C(C=C4)OC(F)(F)F",
    299: "COC1=CC=CC2=C1NC(=C2)C3=C4C(=NC=NN4C(=N3)C5CCC(CC5)C(=O)O)N",
    300: "CC1=CN=C(C=N1)CNC(=O)C2=C3N(C4=CC=CC=C4S3)C5=C(C2=O)C=CC(=N5)N6CCCN(CC6)C",
    301: "CC(C)CC(=O)NC1=NNC2=C1CN(C2(C)C)C(=O)C3CCN(CC3)C",
    302: "C1COCCN1C2=NC(=NC3=C2OC4=C3C=CC=N4)C5=CC(=CC=C5)O",
    303: "CC1=C(SC(=N1)NC(=O)C)C2=CC(=C(C=C2)Cl)S(=O)(=O)NCCO",
    304: "C1=CC=C(C=C1)C2=C(C=CC=N2)C(=O)O",
    305: "C1=CC(=CC=C1C2=CC(=C(S2)NC(=O)N)C(=O)N)F",
    306: "CC1=CN=C(N=C1NC2=CC(=CC=C2)S(=O)(=O)NC(C)(C)C)NC3=CC=C(C=C3)OCCN4CCCC4",
    308: "COC1=CC2=C(C=CN=C2C=C1OCCCN3CCOCC3)OC4=C(C=C(C=C4)NC(=O)C5(CC5)C(=O)NC6=CC=C(C=C6)F)F",
    309: "C[C@H](C1=CC=C(C=C1)C(=O)NC2=C3C=CNC3=NC=C2)N.Cl",
    310: "C1COCCN1C2=NC(=NC3=C2OC4=C3C=CC=N4)C5=CC(=CC=C5)NC(=O)C6=CN=C(C=C6)N",
    312: "CC1=CC(=NO1)NC(=O)NC2=C(C=C(C=C2)OC3=C4C=C(C(=CC4=NC=C3)OC)OC)Cl",
    317: "CNC(=O)NC1=CC=C(C=C1)C2=NC3=C(C=NN3C4CCC5(CC4)OCCO5)C(=N2)N6CC7CCC(C6)O7",
    326: "CCN1C2=C(C(=NC=C2OC[C@H]3CCCNC3)C#CC(C)(C)O)N=C1C4=NON=C4N",
    328: "CC1(CC2=C(C(=O)C1)C(=NN2C3=CC(=C(C=C3)C(=O)N)NC4CCC(CC4)O)C(F)(F)F)C",
    329: "CC1=C(C=C(C=C1)C(=O)NC2=CC(=CC(=C2)NC(=O)C=C)C(F)(F)F)NC(=O)C3=CC=NO3",
    330: "C1CC1C(=O)NC2=NNC3=C2C=CC(=C3)C4=CC(=CC=C4)C(=O)NC5CC5",
    331: "CC1=C(C=C(C=C1)N2C(=O)C=CC3=CN=C4C=CC(=CC4=C32)C5=CNN=C5)NC(=O)C=C",
    333: "C1=CC=C(C=C1)S(=O)(=O)N(CC(F)(F)F)C2=CC=C(C=C2)C(C(F)(F)F)(C(F)(F)F)O",
    341: "C1CC(C2=C(C1)C3=C(N2)C=CC(=C3)Cl)C(=O)N",
    342: "CC(C)(C)C1=CC=C(C=C1)C(=O)NC(=S)NC2=CC=C(C=C2)NC(=O)CCCCN(C)C",
    345: "CC1=C(C=C(C=C1)NC2=NC=NC(=C2)C3=CC(=CC=C3)N4C(=O)C5=CC=CC=C5C4=O)NS(=O)(=O)C",
    356: "C1CNCCC1(C2=CC=C(C=C2)C3=CNN=C3)C4=CC=C(C=C4)Cl",
    362: "C=CC(=O)NC1=C(C=C2C(=C1)C(=NC=N2)NC3=CC(=C(C=C3)F)Cl)OCCCN4CCOCC4",
    363: "COC1=C(C=C2C(=C1)N=CN=C2NC3=CC(=C(C=C3)F)Cl)NC(=O)/C=C/CN4CCCCC4",
    366: "C1=CC(=CC(=C1)N)C2=CC3=C(N2)N=CN=C3OC4=CC=CC(=C4)O",
    371: "C1=CC(=CC(=C1)N2C(=O)C=CC3=CN=C4C=CC(=CC4=C32)C5=CN=C(C=C5)N)C(F)(F)F",
    372: "CC(C)(C(=O)NC1=CC(=CC=C1)S(=O)(=O)NC2=NC3=CC=CC=C3N=C2NC4=C(C=CC(=C4)OC)Cl)N",
    374: "C1=CC2=NC=CC(=C2C=C1/C=C\\3/C(=O)NC(=O)S3)C4=CC=NC=C4",
    375: "CCN1C2=NC(=NC(=C2C=C(C1=O)C3=CC=NN3)C)N",
    380: "CN1CCN(CC1)CC(=O)N(C)C2=CC=C(C=C2)N=C(C3=CC=CC=C3)C4=C(NC5=C4C=CC(=C5)C(=O)OC)O",
    381: "C=CC(=O)NC1=CC2=C(C=C1)N=CN=C2NC3=CC(=C(C=C3)OCC4=CC(=CC=C4)F)Cl",
    382: "CC1=C(SC2=C1N=C(N=C2N3CCOCC3)C4=CN=C(N=C4)N)CN5CCN(CC5)C(=O)[C@H](C)O",
    406: "CCNC(=O)NC1=NC=C(S1)C2=CC(=NC(=N2)C3=NC=CN=C3)C4=C(C=C(C=C4C)OC)C",
    407: "C1CSC2=C(C(=O)N1)SC3=C2C=C(C=C3)O",
    408: "CC1=CC(=CC(=C1)OCC2=CC=C(C=C2)CN3CCC[C@@H]3CO)CS(=O)(=O)C4=CC=CC=C4",
    412: "CC1(CCC(=C(C1)C2=CC=C(C=C2)Cl)CN3CCN(CC3)C4=CC(=C(C=C4)C(=O)NS(=O)(=O)C5=CC(=C(C=C5)NCC6CCOCC6)[N+](=O)[O-])OC7=CN=C8C(=C7)C=CN8)C",
    415: "C1=CC=C(C=C1)CSCCC(CCCCC(=O)O)SCC2=CC=CC=C2",
    416: "CC1=NN=C(O1)C2=NN=C(C=C2)N3CCC(CC3)OC4=C(C=CC(=C4)F)Cl",
    427: "C1=NC2=C(N1[C@H]3[C@H]([C@@H]([C@H](O3)CO)O)O)N=C(NC2=O)N",
    428: "C1=CC(=CC=C1CCC2=CNC3=C2C(=O)NC(=N3)N)C(=O)N[C@@H](CCC(=O)O)C(=O)O",
    431: "COC1=C(C(=CC=C1)F)C2=NCC3=CN=C(N=C3C4=C2C=C(C=C4)Cl)NC5=CC(=C(C=C5)C(=O)O)OC",
    432: "CN1CC[C@@H]([C@@H](C1)O)C2=C(C=C(C3=C2OC(=CC3=O)C4=CC=CC=C4Cl)O)O",
    435: "CCCCCCCCC1C(C(=C)C(=O)O1)C(=O)O",
    437: "C[C@H](/C=C(\\C)/C=C/C(=O)NO)C(=O)C1=CC=C(C=C1)N(C)C",
    438: "CC1=C(C2=CC=CC=C2N1)CCNCC3=CC=C(C=C3)/C=C/C(=O)NO",
    439: "C[C@@H](C(=O)N[C@@H](C1CCCCC1)C(=O)N2CCC[C@H]2C3=NC(=CS3)C(=O)C4=CC=C(C=C4)F)NC",
    442: "C1=CC(=C(C=C1Cl)C(=O)NC2=CC(=CC(=C2)C(F)(F)F)C(F)(F)F)O",
    446: "CC1=CSC(=NC2CCCCC2)N1/N=C/C3=C(C(=C(C=C3)O)O)O",
    447: "C1CNCCC1C2=CC(=NN2)C3=CC=NC=C3",
    449: "C1CCC(C1)C2=CC(=NN2)NC3=NC(=NC=C3)NC4=CC=C(C=C4)NC(=O)NC5=CC=CC(=C5)C(F)(F)F",
    461: "C/C(=N\\NC(=S)N1CCC1)/C2=CC=CC=N2",
    474: "CN(C(=O)N1[C@](SC(=N1)C2=C(C=CC(=C2)F)F)(CCCN)C3=CC=CC=C3)OC",
    476: "CC1=NC(=CC=C1)C2=C(N=C(N2)C(C)(C)C)C3=CC4=C(C=C3)OCO4",
    477: "CC1=NC(=CC=C1)C2=NN(C=C2C3=CC=NC4=CC=CC=C34)C(=S)NC5=CC=CC=C5",
    478: "C1CN(CCN1)C2=CC=C(C=C2)C3=CN4C(=C(C=N4)C5=CC=NC6=CC=CC=C56)N=C3",
    546: "CCCCCCCCC1=CC=C(C=C1)CCC(CO)(CO)N.Cl",
    552: "CCOC1=CC=CC=C1N=NC2=C(NN(C2=O)C3=NC(=CS3)C4=CC=CC=C4)C",
    562: "C1CC1NS(=O)(=O)C2=CC(=C(C=C2)C3=CSC=C3)NC(=O)NC4=CC=CC(=C4)C(F)(F)F",
    563: "C1=CC(=CC=C1NC(=S)NNC2=C(C=C(C=C2[N+](=O)[O-])C(F)(F)F)[N+](=O)[O-])F",
    573: "CC1=CC(=CN=C1C2=CC(=NC=C2)C)CC(=O)NC3=NC=C(C=C3)C4=NC=CN=C4",
    574: "CC1=NC=CC(=C1)C2=CC=C(C=C2)CC(=O)NC3=CC=C(C=C3)C4=CN=CC=C4",
    576: "CCC(C)CNCC(=O)N1CCC2=C(C1COC3=CC=CC(=C3)C)C=CS2",
    1001: "C1=NC(=C(N1[C@H]2[C@@H]([C@@H]([C@H](O2)COP(=O)(O)O)O)O)N)C(=O)N",
    1003: "CC[C@@]1(C2=C(COC1=O)C(=O)N3CC4=CC5=CC=CC=C5N=C4C3=C2)O",
    1004: "CC[C@@]1(C[C@H]2C[C@@](C3=C(CCN(C2)C1)C4=CC=CC=C4N3)(C5=C(C=C6C(=C5)[C@]78CCN9[C@H]7[C@@](C=CC9)([C@H]([C@@]([C@@H]8N6C)(C(=O)OC)O)OC(=O)C)CC)OC)C(=O)OC)O",
    1005: "N.N.Cl[Pt]Cl",
    1006: "C1=CN(C(=O)N=C1N)[C@H]2[C@H]([C@@H]([C@H](O2)CO)O)O",
    1007: "CC1=C2[C@H](C(=O)[C@@]3([C@H](C[C@@H]4[C@]([C@H]3[C@@H]([C@@](C2(C)C)(C[C@@H]1OC(=O)[C@@H]([C@H](C5=CC=CC=C5)NC(=O)OC(C)(C)C)O)O)OC(=O)C6=CC=CC=C6)(CO4)OC(=O)C)O)C)O",
    1008: "CN(CC1=CN=C2C(=N1)C(=NC(=N2)N)N)C3=CC=C(C=C3)C(=O)N[C@@H](CCC(=O)O)C(=O)O",
    1009: "CC1=C(C(CCC1)(C)C)/C=C/C(=C/C=C/C(=C/C(=O)O)/C)/C",
    1010: "COC1=C(C=C2C(=C1)N=CN=C2NC3=CC(=C(C=C3)F)Cl)OCCCN4CCOCC4",
    1011: "CC1(CCC(=C(C1)CN2CCN(CC2)C3=CC=C(C=C3)C(=O)NS(=O)(=O)C4=CC(=C(C=C4)N[C@H](CCN5CCOCC5)CSC6=CC=CC=C6)S(=O)(=O)C(F)(F)F)C7=CC=C(C=C7)Cl)C",
    1012: "C1=CC=C(C=C1)NC(=O)CCCCCCC(=O)NO",
    1013: "CC1=C(C=C(C=C1)C(=O)NC2=CC(=CC(=C2)C(F)(F)F)N3C=C(N=C3)C)NC4=NC=CC(=N4)C5=CN=CC=C5",
    1014: "COC1=CC(=C(C(=C1NS(=O)(=O)C2(CC2)C[C@@H](CO)O)NC3=C(C=C(C=C3)I)F)F)F",
    1015: "C1CC1CONC(=O)C2=C(C(=C(C=C2)F)F)NC3=C(C=C(C=C3)I)Cl",
    1016: "C[C@@H]1CC[C@H]2C[C@@H](/C(=C/C=C/C=C/[C@H](C[C@H](C(=O)[C@@H]([C@@H](/C(=C/[C@H](C(=O)C[C@H](OC(=O)[C@@H]3CCCCN3C(=O)C(=O)[C@@]1(O2)O)[C@H](C)C[C@@H]4CC[C@H]([C@@H](C4)OC)OC(=O)C(C)(CO)CO)C)/C)O)OC)C)C)/C)OC",
    1017: "C1CC1C(=O)N2CCN(CC2)C(=O)C3=C(C=CC(=C3)CC4=NNC(=O)C5=CC=CC=C54)F",
    1018: "C[C@@]1(CCCN1)C2=NC3=C(C=CC=C3N2)C(=O)N",
    1019: "CN1CCN(CC1)CCCOC2=C(C=C3C(=C2)N=CC(=C3NC4=CC(=C(C=C4Cl)Cl)OC)C#N)OC",
    1020: "C1CC(=O)NC(=O)C1N2CC3=C(C2=O)C=CC=C3N",
    1021: "CNC(=O)C1=CC=CC=C1SC2=CC3=C(C=C2)C(=NN3)/C=C/C4=CC=CC=N4",
    1022: "C1C[C@@H](CNC1)NC(=O)C2=C(C=C(S2)C3=CC(=CC=C3)F)NC(=O)N",
    1023: "CN1C=C(C2=CC=CC=C21)/C=C\\3/C4=C(C=CC=N4)NC3=O.Cl",
    1024: "C[C@@]12[C@](C[C@@H](O1)N3C4=CC=CC=C4C5=C6C(=C7C8=CC=CC=C8N2C7=C53)CNC6=O)(CO)O",
    1025: "CN1C=C(C2=CC=CC=C21)C3=C(C(=O)NC3=O)C4=C(C=C(C=C4)Cl)Cl",
    1026: "C[C@H]1C[C@@H]([C@@H]([C@H](/C=C(/[C@@H]([C@H](/C=C\\C=C(\\C(=O)NC2=CC(=O)C(=C(C1)C2=O)NCC=C)/C)OC)OC(=O)N)\\C)C)O)OC",
    1028: "C1=CC(=C(C(=C1)F)N(C2=NC(=C(C=C2)C(=O)N)C3=C(C=C(C=C3)F)F)C(=O)N)F",
    1029: "CC1(CNC2=C1C=CC(=C2)NC(=O)C3=C(N=CC=C3)NCC4=CC=NC=C4)C",
    1030: "C1COCCN1C2=CC(=O)C=C(O2)C3=C4C(=CC=C3)SC5=CC=CC=C5S4",
    1031: "CN(C(=S)C1=CC=CC=C1)NC(=O)CC(=O)NN(C)C(=S)C2=CC=CC=C2",
    1032: "CN(C)C/C=C/C(=O)NC1=C(C=C2C(=C1)C(=NC=N2)NC3=CC(=C(C=C3)F)Cl)O[C@H]4CCOC4",
    1033: "CS(=O)(=O)C1=CC(=C(C=C1)C(=O)NC2=CC(=C(C=C2)Cl)C3=CC=CC=N3)Cl",
    1034: "C[C@@]12[C@@H]([C@@H](C[C@@H](O1)N3C4=CC=CC=C4C5=C6C(=C7C8=CC=CC=C8N2C7=C53)CNC6=O)NC)OC",
    1036: "CCCS(=O)(=O)NC1=C(C(=C(C=C1)F)C(=O)C2=CNC3=C2C=C(C=N3)Cl)F",
    1037: "C1CCN(C1)C(=O)NC2=CC=CC(=C2)NC3=NC=C(C(=N3)NCCCNC(=O)C4=CC=CS4)I",
    1038: "C1COCCN1C2=CC(=O)C3=C(O2)C(=CC=C3)C4=CC=CC5=C4SC6=CC=CC=C56",
    1039: "C[C@H]1[C@@H]([C@H]([C@H]([C@@H](O1)OC2=C(OC3=CC(=CC(=C3C2=O)O)O)C4=CC=C(C=C4)O)O)OC(=O)C)OC(=O)C",
    1042: "CC1=CC=C(C=C1)N2C(=CC(=N2)C(C)(C)C)NC(=O)NC3=CC=C(C4=CC=CC=C43)OCCN5CCOCC5",
    1043: "CCOC1=C(C(=CC(=N1)NC(=O)CC2=C(C=CC(=C2)OC)OC)N)C#N",
    1046: "C1=CC=C(C(=C1)C2=CC3=C(C4=C(N3)C=CC(=C4)O)C5=C2C(=O)NC5=O)Cl",
    1047: "CC(C)OC1=C(C=CC(=C1)OC)C2=N[C@H]([C@H](N2C(=O)N3CCNC(=O)C3)C4=CC=C(C=C4)Cl)C5=CC=C(C=C5)Cl",
    1048: "C1=CC(=CC=C1/C=C\\2/C(=O)N=C(S2)N)O",
    1049: "CCN(CC)CCCCNC1=NC2=NC(=C(C=C2C=N1)C3=CC(=CC(=C3)OC)OC)NC(=O)NC(C)(C)C",
    1050: "COC1=C(C=C2C(=C1)C(=NC=N2)NC3=CC=C(C=C3)NC(=O)C4=CC=CC=C4)OCCCN5CCOCC5",
    1051: "COC1=C(C(=CC=C1)F)C2=NCC3=CN=C(N=C3C4=C2C=C(C=C4)Cl)NC5=CC(=C(C=C5)C(=O)O)OC",
    1052: "C1=CC2=C(C=CC(=C2)C=C3C(=O)NC(=NCC4=CC=CS4)S3)N=C1",
    1053: "C1CC(C1)(C2=CC=C(C=C2)C3=C(C=C4C(=N3)C=CN5C4=NNC5=O)C6=CC=CC=C6)N",
    1054: "CC1=C(C(=O)N(C2=NC(=NC=C12)NC3=NC=C(C=C3)N4CCNCC4)C5CCCC5)C(=O)C",
    1057: "CC(C)(C#N)C1=CC=C(C=C1)N2C3=C4C=C(C=CC4=NC=C3N(C2=O)C)C5=CC6=CC=CC=C6N=C5",
    1058: "CS(=O)(=O)N1CCN(CC1)CC2=CC3=C(S2)C(=NC(=N3)C4=C5C=NNC5=CC=C4)N6CCOCC6",
    1059: "C[C@H]1COCCN1C2=NC(=NC3=C2C=CC(=N3)C4=CC(=C(C=C4)OC)CO)N5CCOC[C@@H]5C",
    1060: "C1=CC(=C(C=C1I)F)NC2=C(C=CC(=C2F)F)C(=O)NOC[C@@H](CO)O",
    1061: "CN(C)CCOC1=CC=C(C=C1)C2=NC(=C(N2)C3=CC=NC=C3)C4=CC5=C(C=C4)C(=NO)CC5",
    1062: "CN1C=NC2=C1C=C(C(=C2F)NC3=C(C=C(C=C3)Br)Cl)C(=O)NOCCO",
    1066: "CC1=CN2C(=O)C=C(N=C2C(=C1)[C@@H](C)NC3=CC=CC=C3C(=O)O)N4CCOCC4",
    1067: "C1/C(=C\\C2=CC=CS2)/C(=O)/C(=C/C3=CC=CS3)/C1",
    1068: "CC1=CC(=C(N1)/C=C\\2/C(=C/C(=C/3\\C=C4C=CC=CC4=N3)/N2)OC)C.CS(=O)(=O)O",
    1069: "C1COCCN1CC2=CC(=O)C(=CO2)OCCCCCSC3=C4C=CC(=CC4=NC=C3)C(F)(F)F.Cl.Cl",
    1070: "CC1=CC(=C(C=C1)C2=C(C=CN=C2)C3=CN=C(S3)NC4=CC=C(C=C4)O)Cl",
    1072: "C1=CC(=CC=C1S(=O)(=O)N(CC2=C(C=C(C=C2)C3=NOC=N3)F)[C@H](CCC(F)(F)F)C(=O)N)Cl",
    1073: "C1=C(C(=O)NC(=O)N1)F",
    1079: "CC1=C(C(=CC=C1)Cl)NC(=O)C2=CN=C(S2)NC3=CC(=NC(=N3)C)N4CCN(CC4)CCO",
    1080: "CC1=C2[C@H](C(=O)[C@@]3([C@H](C[C@@H]4[C@]([C@H]3[C@@H]([C@@](C2(C)C)(C[C@@H]1OC(=O)[C@@H]([C@H](C5=CC=CC=C5)NC(=O)C6=CC=CC=C6)O)O)OC(=O)C7=CC=CC=C7)(CO4)OC(=O)C)O)C)OC(=O)C",
    1083: "C[C@H](C1=C(C=CC(=C1Cl)F)Cl)OC2=C(N=CC(=C2)C3=CN(N=C3)C4CCNCC4)N",
    1084: "C[C@@H]1CC[C@H]2C[C@@H](/C(=C/C=C/C=C/[C@H](C[C@H](C(=O)[C@@H]([C@@H](/C(=C/[C@H](C(=O)C[C@H](OC(=O)[C@@H]3CCCCN3C(=O)C(=O)[C@@]1(O2)O)[C@H](C)C[C@@H]4CC[C@H]([C@@H](C4)OC)O)C)/C)O)OC)C)C)/C)OC",
    1085: "CNC(=O)C1=NC=CC(=C1)OC2=CC=C(C=C2)NC(=O)NC3=CC(=C(C=C3)Cl)C(F)(F)F",
    1086: "CC[C@@H]1C(=O)N(C2=CN=C(N=C2N1C3CCCC3)NC4=C(C=C(C=C4)C(=O)NC5CCN(CC5)C)OC)C",
    1088: "CCC1=C2CN3C(=CC4=C(C3=O)COC(=O)[C@@]4(CC)O)C2=NC5=C1C=C(C=C5)OC(=O)N6CCC(CC6)N7CCCCC7",
    1089: "C1CC[C@H]([C@@H](C1)[NH-])[NH-].C(=O)(C(=O)O)O.[Pt+2]",
    1091: "CC1=CC(=CC2=C1N=C(N2)C3=C(C=CNC3=O)NC[C@H](C4=CC(=CC=C4)Cl)O)N5CCOCC5",
    1093: "CCC1=CC(=C(C=C1N2CCC(CC2)N3CCN(CC3)S(=O)(=O)C)OC)NC4=NC=CC(=N4)C5=C(N=C6N5C=CC=C6)C7=CC(=C(C=C7)OC)C(=O)NC8=C(C=CC=C8F)F",
    1096: "CC1=CC(=NN1)NC2=CC(=NC(=N2)SC3=CC=C(C=C3)NC(=O)C4CC4)N5CCN(CC5)C",
    1114: "CC1(C(=C)N(C2=CC=CC=C21)CCCCCC(=O)O)C",
    1129: "CCC1=CN=CN=C1N2CCN(CC2)CC3=NC4=C(N3)C=C(C=C4)C(F)(F)F",
    1131: "COCC1(C(=O)C2CCN1CC2)CO",
    1133: "C1=CC=C2C(=C1)C(=CN2)CCNC3=CC=C(C=C3)NC4=CC=NC=C4",
    1135: "C[C@@H]1CN(C[C@@H](N1)C)C2=CC=C(C=C2)C(=O)NC3=NNC(=C3)CCC4=CC(=CC(=C4)OC)OC",
    1136: "C1CN(CCC1(C(=O)N[C@@H](CCO)C2=CC=C(C=C2)Cl)N)C3=NC=NC4=C3C=CN4",
    1142: "CC(C)S(=O)(=O)C1=CC=CC=C1NC2=CC(=NC3=C2C=CN3)NC4=NC=C(C=C4)C(=O)NC",
    1143: "CC(C)S(=O)(=O)C1=CC=CC=C1NC2=NC(=NC(=C2Cl)N)NC3=C(C=C(C=C3)N4CCC(CC4)C(=O)N)OC",
    1149: "CC(C)C1=CC=CC=C1CC2=CC(=C(C(=C2O)O)O)C(=O)NC3=CC=C(C=C3)S(=O)(=O)C4=CC=CC=C4C(C)(C)C",
    1158: "CC(C)N1C(=O)CCN(C2=NC(=NC=C21)NC3=C(C=C(C=C3)C(=O)NC4CCN(CC4)C)OC)C5CCCC5",
    1161: "CN(C)C/C=C/C(=O)NC1=CC=C(C=C1)C(=O)NC2=CC=CC(=C2)NC3=NC=CC(=N3)C4=CN=CC=C4",
    1164: "CCOC1=C(C=CC(=C1)N2CCC(CC2)O)NC3=NC=C4C(=N3)N(C5=CC=CC=C5C(=O)N4C)C",
    1167: "CCN1CCN(CC1)CC2=C(C=C(C=C2)NC(=O)C3=CC(=C(C=C3)C)OC4=C5C=CNC5=NC=C4)C(F)(F)F",
    1168: "COCCOC1=C(C=C2C(=C1)C(=NC=N2)NC3=CC=CC(=C3)C#C)OCCOC",
    1170: "CCC1=CC(=C(C=C1O)O)C2=NNC(=C2C3=CC4=C(C=C3)OCCO4)C",
    1175: "CNCC1=CC=C(C=C1)C2=C3CCNC(=O)C4=C3C(=CC(=C4)F)N2",
    1177: "C1C[C@H](CNC1)C2=CC=C(C=C2)N3C=C4C=CC=C(C4=N3)C(=O)N",
    1179: "CC(C)(C1=NC(=CC=C1)N2C3=NC(=NC=C3C(=O)N2CC=C)NC4=CC=C(C=C4)N5CCN(CC5)C)O",
    1180: "CCC1=C2N=C(C=C(N2N=C1)NCC3=C[N+](=CC=C3)[O-])N4CCCC[C@H]4CCO",
    1184: "C[C@@H]1COCCN1C2=NC(=NC(=C2)C3(CC3)S(=O)(=O)C)C4=C5C=CNC5=CC=C4",
    1185: "C[C@@H]1CN(C[C@@H](O1)C)CC(=O)NC2=CC3=C(C=C2)SC4=C(C3)C=CC=C4C5=CC(=O)C=C(O5)N6CCOCC6",
    1187: "CC1=C(C=C(C=C1)NC(=O)C2=CC3=C(C=C2)OCCO3)NC(=O)C4=CC5=C(C=C4)N=C(C=C5)C",
    1190: "C1=CN(C(=O)N=C1N)[C@H]2C([C@@H]([C@H](O2)CO)O)(F)F",
    1191: "B([C@H](CC(C)C)NC(=O)[C@H](CC1=CC=CC=C1)NC(=O)C2=NC=CN=C2)(O)O",
    1192: "CCN1C2=CC(=NC=C2N=C1C3=NON=C3N)OC4=CC=CC(=C4)NC(=O)C5=CC=C(C=C5)OCCN6CCOCC6",
    1194: "CC1=NC(=CC=C1)C2=C(N=C(N2)C(C)(C)C)C3=CC4=C(C=C3)OCO4",
    1199: "CC/C(=C(\\C1=CC=CC=C1)/C2=CC=C(C=C2)OCCN(C)C)/C3=CC=CC=C3",
    1200: "C[C@]12CC[C@H]3[C@H]([C@@H]1CC[C@@H]2O)[C@@H](CC4=C3C=CC(=C4)O)CCCCCCCCCS(=O)CCCC(C(F)(F)F)(F)F",
    1202: "C1=CC(=CC(=C1)N2C(=O)C=CC3=CN=C4C=CC(=CC4=C32)C5=CN=C(C=C5)N)C(F)(F)F",
    1218: "CC1=C(SC2=C1C(=N[C@H](C3=NN=C(N32)C)CC(=O)OC(C)(C)C)C4=CC=C(C=C4)Cl)C",
    1219: "CN1CC2=C(C=CC(=C2)NS(=O)(=O)C3=CC=CC=C3OC)NC1=O",
    1230: "C1=CC=C(C=C1)CN2C3=CC=CC=C3C(=C(C2=O)C(=O)NCC(=O)O)O",
    1232: "CC1=C(C(=NO1)C)C2=C(C=C3C(=C2)N=CC4=C3N(C(=O)N4)[C@H](C)C5=CC=CC=N5)OC",
    1236: "CC(C)N1CCC(CC1)NC2=NC(=NC3=CC(=C(C=C32)OC)OCCCN4CCCC4)C5CCCCC5",
    1237: "CC(C)N(CCCNC(=O)NC1=CC=C(C=C1)C(C)(C)C)C[C@@H]2[C@H]([C@H]([C@@H](O2)N3C=CC4=C(N=CN=C43)N)O)O",
    1239: "COC1=CC=C(C=C1)C(=O)CC2(C3=C(C=CC(=C3NC2=O)Cl)Cl)O",
    1241: "CC1=CN=C(N1)C2=CN=C(N=C2C3=C(C=C(C=C3)Cl)Cl)NCCNC4=NC=C(C=C4)C#N",
    1242: "C[C@H]1C/C=C\\C(=O)[C@H]([C@H](C/C=C/C2=C(C(=CC(=C2)OC)O)C(=O)O1)O)O",
    1243: "COC1=CC(=CC(=C1OC)OC)/C=C/C(=O)N2CCC=CC2=O",
    1248: "C1CN(CCC1CCCCNC(=O)/C=C/C2=CN=CC=C2)C(=O)C3=CC=CC=C3",
    1249: "CC1=CC2=C(C=C1)N=C(C3=NC=C(N23)C)NCCN.Cl",
    1250: "CC1=CC(=NN1)NC2=C(C=C(C(=N2)N[C@@H](C)C3=CC=C(C=C3)F)C#N)F",
    1254: "CCOC(=O)CCNC1=CC(=NC(=N1)C2=CC=CC=N2)N3CCC4=CC=CC=C4CC3",
    1259: "CN1C(=NC=N1)[C@@H]2[C@H](NC3=CC(=CC4=C3C2=NNC4=O)F)C5=CC=C(C=C5)F",
    1262: "C1CCN(C1)C2CCN(CC2)C(=O)C3=CC(=C(C=C3)C(=O)N4CCC(CC4)N5CCCC5)NC6=CC=CC=C6",
    1263: "CC(C)N1CCC(CC1)NC2=NC(=NC3=CC(=C(C=C32)OC)OCCCN4CCCC4)N5CCC(CC5)(F)F",
    1264: "CC(C)N(CCCNC(=O)NC1=CC=C(C=C1)C(C)(C)C)C[C@@H]2[C@H]([C@H]([C@@H](O2)N3C=C(C4=C(N=CN=C43)N)Br)O)O",
    1268: "C1CSCC2=C1N=C(NC2=O)C3=CC=C(C=C3)C(F)(F)F",
    1371: "CCCS(=O)(=O)NC1=C(C(=C(C=C1)F)C(=O)C2=CNC3=C2C=C(C=N3)Cl)F",
    1372: "CC1=C2C(=C(N(C1=O)C)NC3=C(C=C(C=C3)I)F)C(=O)N(C(=O)N2C4=CC=CC(=C4)NC(=O)C)C5CC5",
    1373: "CC(C)(C)C1=NC(=C(S1)C2=NC(=NC=C2)N)C3=C(C(=CC=C3)NS(=O)(=O)C4=C(C=CC=C4F)F)F",
    1375: "CN1C(=O)N2C=NC(=C2N=N1)C(=O)N",
    1377: "CN(C)C/C=C/C(=O)NC1=C(C=C2C(=C1)C(=NC=N2)NC3=CC(=C(C=C3)F)Cl)O[C@H]4CCOC4",
    1378: "CC1=C(N=C(N=C1N)[C@H](CC(=O)N)NC[C@@H](C(=O)N)N)C(=O)N[C@@H]([C@H](C2=CN=CN2)OC3C(C(C(C(O3)CO)O)O)OC4C(C(C(C(O4)CO)O)OC(=O)N)O)C(=O)N[C@H](C)[C@H]([C@H](C)C(=O)N[C@@H]([C@@H](C)O)C(=O)NCCC5=NC(=CS5)C6=NC(=CS6)C(=O)NCCC[S+](C)C)O",
    1382: "CC(=O)N1CCN(CC1)CCOC2=CC=C(C=C2)C3CCN(CC3)C4=NN5C(=NN=C5C(F)(F)F)CC4",
    1386: "C[C@H]1[C@H]([C@H](C[C@@H](O1)O[C@H]2C[C@@](CC3=C2C(=C4C(=C3O)C(=O)C5=C(C4=O)C(=CC=C5)OC)O)(C(=O)CO)O)N)O",
    1392: "CC1=C(N=C(N=C1N)[C@H](CC(=O)N)NC[C@@H](C(=O)N)N)C(=O)N[C@@H]([C@H](C2=CN=CN2)OC3C(C(C(C(O3)CO)O)O)OC4C(C(C(C(O4)CO)O)OC(=O)N)O)C(=O)N[C@H](C)[C@H]([C@H](C)C(=O)N[C@@H]([C@@H](C)O)C(=O)NCCC5=NC(=CS5)C6=NC(=CS6)C(=O)NCCC[S+](C)C)O",
    1393: "C1=CN(C(=O)N=C1N)[C@H]2C([C@@H]([C@H](O2)CO)O)(F)F",
    1394: "C[C@@H]1COCCN1C2=NC(=NC(=C2)C3(CC3)[S@](=N)(=O)C)C4=CN=CC5=C4C=CN5",
    1401: "CC1=NC=C(N1C(C)C)C2=NC(=NC=C2)NC3=CC=C(C=C3)S(=O)(=O)C",
    1402: "C1C[C@@H](CNC1)NC(=O)C2=C(C=C(S2)C3=CC(=CC=C3)F)NC(=O)N",
    1403: "C[C@@H](C1=CN2C=CN=C2C=C1)N3C4=NC(=CN=C4N=N3)C5=CN(N=C5)C",
    1409: "CC1=CC=C(C=C1)C(=O)N(CCCN)C(C2=NC3=C(C(=NS3)C)C(=O)N2CC4=CC=CC=C4)C(C)C",
    1414: "C[C@]12CC[C@H]3[C@H]([C@@H]1CC[C@@H]2O)[C@@H](CC4=C3C=CC(=C4)O)CCCCCCCCCS(=O)CCCC(C(F)(F)F)(F)F",
    1416: "CNC(=O)CN1CCC(CC1)OC2=C(C=C3C(=C2)C(=NC=N3)NC4=C(C(=CC=C4)Cl)F)OC",
    1425: "C1=CC=C(C=C1)CC(=O)NC2=NN=C(S2)CCSCCC3=NN=C(S3)NC(=O)CC4=CC=CC=C4",
    1427: "C[C@@H](C(=O)N[C@@H](C1CCCCC1)C(=O)N2CCC[C@H]2C(=O)N[C@@H]3[C@@H](CC4=CC=CC=C34)OCC#CC#CCO[C@@H]5CC6=CC=CC=C6[C@@H]5NC(=O)[C@@H]7CCCN7C(=O)[C@H](C8CCCCC8)NC(=O)[C@H](C)NC)NC",
    1432: "CC1=CC(=NN1)NC2=NC(=NC=C2Cl)N[C@@H](C)C3=NC=C(C=N3)F",
    1441: "C[C@H]1COCCN1C2=NC(=NC3=C2C=CC(=N3)C4=CC(=CC=C4)C(=O)NC)N5CCOC[C@@H]5C",
    1444: "C[C@H](C1=CC(=CC2=C1OC(=CC2=O)N3CCOCC3)C(=O)N(C)C)NC4=CC(=CC(=C4)F)F",
    1445: "CCN1C(=NC(=N1)C2CCN(CC2)C(=O)CCO)C3=CN=C(C(=N3)C4=NN=C(O4)C(C)(C)C)N",
    1449: "C1C[C@H](CN(C1)C2=C(C=CC=C2C3=CC=CC=C3)/C=C\\4/C(=O)NC(=O)S4)N",
    1463: "CC(C)OC1=NNC(=C1)NC2=NC(=NC=C2Cl)NC(C)C3=NC=C(C=C3)F",
    1490: "CCC1=C2CN3C(=CC4=C(C3=O)COC(=O)[C@@]4(CC)O)C2=NC5=C1C=C(C=C5)O",
    1494: "CCC1=C2CN3C(=CC4=C(C3=O)COC(=O)[C@@]4(CC)O)C2=NC5=C1C=C(C=C5)O",
    1495: "C1CC1C(=O)N2CCN(CC2)C(=O)C3=C(C=CC(=C3)CC4=NNC(=O)C5=CC=CC=C54)F",
    1496: "N.N.Cl[Pt]Cl",
    1497: "C[C@@H]1CN(C[C@@H](N1)C)C2=CC=C(C=C2)C(=O)NC3=NNC(=C3)CCC4=CC(=CC(=C4)OC)OC",
    1498: "CN1C=NC2=C1C=C(C(=C2F)NC3=C(C=C(C=C3)Br)Cl)C(=O)NOCCO",
    1502: "CC(CS(=O)(=O)C1=CC=C(C=C1)F)(C(=O)NC2=CC(=C(C=C2)C#N)C(F)(F)F)O",
    1507: "C1CCC(C1)[C@@H](CC#N)N2C=C(C=N2)C3=C4C=CNC4=NC=N3",
    1510: "CC1(CC(C1)C2=NC(=C3N2C=CN=C3N)C4=CC5=C(C=C4)C=CC(=N5)C6=CC=CC=C6)O",
    1511: "C[C@H]1[C@@H]([C@H](C[C@@H](O1)O[C@H]2C[C@@](CC3=C2C(=C4C(=C3O)C(=O)C5=C(C4=O)C(=CC=C5)OC)O)(C(=O)CO)O)N)O",
    1512: "C1CNP(=O)(OC1)N(CCCl)CCCl",
    1526: "COC1=CC(=C(C(=C1NS(=O)(=O)C2(CC2)C[C@@H](CO)O)NC3=C(C=C(C=C3)I)F)F)F",
    1527: "CS(=O)(=O)N1CCN(CC1)CC2=CC3=C(S2)C(=NC(=N3)C4=C5C=NNC5=CC=C4)N6CCOCC6",
    1529: "C1CC2=CC=CC=C2[C@H]1NC3=C4C=CN(C4=NC=N3)[C@@H]5C[C@H]([C@H](C5)O)COS(=O)(=O)N",
    1530: "C1[C@@H]2CN([C@H]1CN2C3=CC=CC=N3)/C=C/C(=O)C4=CC=CC=C4O",
    1531: "CCC(=O)N1CCOC2=C(C1)C=C(C=C2OC[C@H]3CCCN(C3)C)C4=CC(=C(C=C4)OC)OC",
    1549: "CNC(=O)CN1CCC(CC1)OC2=C(C=C3C(=C2)C(=NC=N3)NC4=C(C(=CC=C4)Cl)F)OC",
    1553: "CN1C(=C(C=N1)Cl)C2=C(OC(=C2)C(=O)N[C@@H](CC3=CC(=C(C=C3)F)F)CN)Cl",
    1557: "C[C@@H](C(=O)N[C@@H](C1CCCCC1)C(=O)N2CCC[C@H]2C3=NC(=CS3)C(=O)C4=CC=C(C=C4)F)NC",
    1558: "CS(=O)(=O)CCNCC1=CC=C(O1)C2=CC3=C(C=C2)N=CN=C3NC4=CC(=C(C=C4)OCC5=CC(=CC=C5)F)Cl",
    1559: "CCNC(=O)C1=NOC(=C1C2=CC=C(C=C2)CN3CCOCC3)C4=CC(=C(C=C4O)O)C(C)C",
    1560: "CC1=C(SC(=N1)NC(=O)N2CCC[C@H]2C(=O)N)C3=CC(=NC=C3)C(C)(C)C(F)(F)F",
    1561: "CC1=NN(C(=N1)C2=CN3CCOC4=C(C3=N2)C=CC(=C4)C5=CN(N=C5)C(C)(C)C(=O)N)C(C)C",
    1563: "CC(C)N(C[C@@H]1[C@H]([C@H]([C@@H](O1)N2C=NC3=C(N=CN=C32)N)O)O)C4CC(C4)CCC5=NC6=C(N5)C=C(C=C6)C(C)(C)C",
    1564: "C1CN(C[C@@H]1C(=O)NC2=CC3=C(C=C2)NN=C3C4=CC=NC=C4)CC(=O)N5CCN(CC5)C6=CC=C(C=C6)C7=NC=CC=N7",
    1576: "CC1=CC2=C(C=C1)N=C(S2)NC(=O)CSC3=NC4=C(C(=O)N3C5=CC=CC=C5)SCC4",
    1578: "CC1=C(C=NO1)C(=O)NC2=CC=C(C=C2)C(F)(F)F",
    1580: "CCCOC1=CC2=C(C=C(N2C=C1)C(=O)C)C3=CC=CC=C3S(=O)(=O)C",
    1581: "CCOC(=O)NC1=CC(=NN2C1=NN=C2C)C3=CC(=C(C=C3)C)NS(=O)(=O)C",
    1582: "CC1=C(C(=NO1)C)C2=CC3=C(C=C2)N(C(=N3)CCC4=CC(=C(C=C4)OC)Cl)C[C@H](C)N5CCOCC5",
    1583: "C1CNCCC1N[C@@H]2C[C@H]2C3=CC=CC=C3",
    1593: "C1=CC=C(C(=C1)N)NC(=O)C2=CC=C(C=C2)CNC(=O)OCC3=CN=CC=C3",
    1594: "COC1=CC=CC2=C1NC(=C2)C3=C4C(=NC=NN4C(=N3)C5CCC(CC5)C(=O)O)N",
    1598: "CC1=CC(=CN=C1C2=CC(=NC=C2)C)CC(=O)NC3=NC=C(C=C3)C4=NC=CN=C4",
    1613: "CC(C)S(=O)(=O)C1=CC=C(C=C1)C2=CN=C(C(=N2)C3=CC(=NO3)C4=CC=C(C=C4)CNC)N",
    1614: "CCC(=O)NC1=CC(=CC=C1)OC2=NC(=NC=C2Cl)NC3=C(C=C(C=C3)N4CCN(CC4)C)OC",
    1615: "CC(C)(C)NS(=O)(=O)C1=CN=CC(=C1)C2=CN3C(=NC(=N3)N)C(=C2)F",
    1617: "C[C@@H](C(=O)N[C@@H](C1CCCCC1)C(=O)N2CCC[C@H]2C(=O)N[C@@H]3[C@@H](CC4=CC=CC=C34)OCC#CC#CCO[C@@H]5CC6=CC=CC=C6[C@@H]5NC(=O)[C@@H]7CCCN7C(=O)[C@H](C8CCCCC8)NC(=O)[C@H](C)NC)NC",
    1618: "CN1C=C(C2=C(N=CN=C21)N)C3=CC4=C(C=C3)N(CC4)C(=O)CC5=CC(=CC=C5)C(F)(F)F",
    1620: "C1[C@@H]2CN([C@H]1CN2C3=CC=CC=N3)/C=C/C(=O)C4=CC=CC=C4O",
    1621: "COC1=CC=C(C=C1)CN2C=CC3=C2C=C(C=C3)C(=O)NO",
    1622: "CC1=NC=CC(=C1)C2=CC=C(C=C2)CC(=O)NC3=CC=C(C=C3)C4=CN=CC=C4",
    1624: "CCNC(=O)C[C@H]1C2=NN=C(N2C3=C(C=C(C=C3)OC)C(=N1)C4=CC=C(C=C4)Cl)C",
    1625: "CC1=CC(=CC(=C1OCCO)C)C2=NC3=C(C(=CC(=C3)OC)OC)C(=O)N2",
    1626: "CC1=C(SC2=C1C(=N[C@H](C3=NN=C(N32)C)CC(=O)NC4=CC=C(C=C4)O)C5=CC=C(C=C5)Cl)C",
    1627: "CCCC1=C(C(=O)NC(=C1)C)CNC(=O)C2=C3C=NN(C3=CC(=C2)C4=CC(=NC=C4)N5CCN(CC5)C)C(C)C",
    1629: "CC1=CN=C(N=C1NCC2=CC=C(C=C2)N3C=CN=N3)C4=CC=CC=C4C(C)C",
    1630: "C1COCCN1C2=CC=C(C=C2)NC3=NC(=CN4C3=NC=C4)C5=CC6=C(C=C5)C=NN6",
    1631: "C1CC[C@H]([C@H](C1)N)NC2=NC=C(C(=N2)NC3=CC(=CC=C3)N4N=CC=N4)C(=O)N",
    1632: "CN(C)C(=O)C1=CC2=CN=C(N=C2N1C3CCCC3)NC4=NC=C(C=C4)N5CCNCC5",
    1634: "C1CC1NS(=O)(=O)C2=CC(=C(C=C2)C3=CSC=C3)NC(=O)NC4=CC=CC(=C4)C(F)(F)F",
    1635: "C1=CC=NC(=C1)C(=O)[O-]",
    1706: "C[C@@H]1C(=O)N(CCN1CCOC2=CC=C(C=C2)C3CCN(CC3)C4=NN5C(=NN=C5OC)C=C4)C",
    1720: "CC1=C2C(=NN1C)CSCC3=NN(C(=C3)CSC4=CC5=CC=CC=C5C(=C4)OCCCC6=C(N(C7=C6C=CC(=C27)Cl)C)C(=O)O)C",
    1736: "CN1C=NC2=C1C=C(C(=C2F)NC3=C(C=C(C=C3)Br)Cl)C(=O)NOCCO",
    1776: "CC1=NN(C(=C1)NC2=NC=C(C(=C2)NC3=CC=CC=C3C(=O)NOC)Cl)C(C)C",
    1778: "CN1C(=C(C=N1)Cl)C2=C(SC(=C2)C(=O)N[C@@H](CC3=CC(=CC=C3)F)CN)Cl.Cl",
    1779: "C1=CC(=C(C(=C1)Cl)N=C2NC(=O)/C(=C/C3=CC4=NC=CN=C4C=C3)/S2)Cl",
    1782: "CC1=C(C=C(C=N1)Cl)NCC2=CC=C(S2)C(=O)N[C@@H](CC3CCCC3)C(=O)NC4CC4",
    1786: "C[C@@H]1CN(C[C@@H](N1)C)C2=CC=C(C=C2)C(=O)NC3=NNC(=C3)CCC4=CC(=CC(=C4)OC)OC",
    1799: "C=CC(=O)N1CCC[C@H](C1)N2C3=NC=NC(=C3C(=N2)C4=CC=C(C=C4)OC5=CC=CC=C5)N",
    1802: "C1=CN(C=N1)CC(O)(P(=O)(O)O)P(=O)(O)O",
    1803: "CC(=O)OC1=CC=C(C=C1)C2(C3=CC=CC=C3NC2=O)C4=CC=C(C=C4)OC(=O)C",
    1804: "CC(=O)OC1=CC=C(C=C1)C2(C3=CC=CC=C3NC2=O)C4=CC=C(C=C4)OC(=O)C",
    1806: "C1CC[C@H]([C@@H](C1)[NH-])[NH-].C(=O)(C(=O)O)O.[Pt+2]",
    1808: "CC[C@@]1(C2=C(COC1=O)C(=O)N3CC4=CC5=C(C=CC(=C5CN(C)C)O)N=C4C3=C2)O",
    1809: "COC1=CC(=CC(=C1O)OC)[C@H]2[C@@H]3[C@H](COC3=O)[C@@H](C4=CC5=C(C=C24)OCO5)O[C@H]6[C@@H]([C@H]([C@H]7[C@H](O6)CO[C@H](O7)C8=CC=CS8)O)O",
    1810: "C1=CC(=C2C(=C1NCCNCCO)C(=O)C3=C(C=CC(=C3C2=O)O)O)NCCNCCO",
    1811: "C[C@@H]1[C@@H](C(=O)N[C@@H](C(=O)N2CCC[C@H]2C(=O)N(CC(=O)N([C@H](C(=O)O1)C(C)C)C)C)C(C)C)NC(=O)C3=C4C(=C(C=C3)C)OC5=C(C(=O)C(=C(C5=N4)C(=O)N[C@H]6[C@H](OC(=O)[C@@H](N(C(=O)CN(C(=O)[C@@H]7CCCN7C(=O)[C@H](NC6=O)C(C)C)C)C)C(C)C)C)N)C",
    1812: "CC1=C(N=C(N=C1N)[C@H](CC(=O)N)NC[C@@H](C(=O)N)N)C(=O)N[C@@H]([C@H](C2=CN=CN2)OC3C(C(C(C(O3)CO)O)O)OC4C(C(C(C(O4)CO)O)OC(=O)N)O)C(=O)N[C@H](C)[C@H]([C@H](C)C(=O)N[C@@H]([C@@H](C)O)C(=O)NCCC5=NC(=CS5)C6=NC(=CS6)C(=O)NCCC[S+](C)C)O",
    1813: "C1=NC2=C(N=C(N=C2N1[C@H]3[C@H]([C@@H]([C@H](O3)CO)O)O)F)N",
    1814: "COC1=NC(=NC2=C1N=CN2[C@H]3[C@H]([C@@H]([C@H](O3)CO)O)O)N",
    1815: "CN(C)/N=N/C1=C(NC=N1)C(=O)N",
    1816: "C[C@]12CC[C@H]3[C@H]([C@@H]1CC[C@@H]2O)[C@@H](CC4=C3C=CC(=C4)O)CCCCCCCCCS(=O)CCCC(C(F)(F)F)(F)F",
    1817: "C/C=C\\1/C(=O)N[C@H](C(=O)O[C@H]\\2CC(=O)N[C@@H](C(=O)N[C@H](CSSCC/C=C2)C(=O)N1)C(C)C)C(C)C",
    1819: "CC1=C2[C@H](C(=O)[C@@]3([C@H](C[C@@H]4[C@]([C@H]3[C@@H]([C@@](C2(C)C)(C[C@@H]1OC(=O)[C@@H]([C@H](C5=CC=CC=C5)NC(=O)OC(C)(C)C)O)O)OC(=O)C6=CC=CC=C6)(CO4)OC(=O)C)O)C)O",
    1825: "COC1=CC(=CC(=C1OC)OC)[C@H]2[C@@H]3[C@H](COC3=O)C(C4=CC5=C(C=C24)OCO5)Br",
    1827: "CC(C)[C@H]1CC2=C(O1)C=CC3=C2O[C@@H]4COC5=CC(=C(C=C5[C@@H]4C3=O)OC)OC",
    1830: "COC1=C(C(=C2C(=C1)C(=O)NC(N2)(NC(=O)OC)NC(=O)OC)OC)OC",
    1831: "C1CNC(=O)C2=C1C3=CC=CC=C3N2",
    1835: "CC(=CC(=O)O[C@H]1CC2=C[C@@H](C[C@@]3([C@H](O3)[C@@H]4[C@@H]1C(=C)C(=O)O4)C)OC2=O)C",
    1838: "C/C/1=C\\CC[C@]([C@H]2C[C@@H](CC[C@]3([C@@H](O3)CC1)C)C(=C)C(=O)O2)(C)O",
    1843: "COC1=C(C=C2C(=C1)C3=C(C4=CC5=C(C=C4C3=O)OCO5)N(C2=O)CCCNCCO)OC",
    1845: "CC(=CCC/C(=C/CC1=C(C=C(C=C1O)/C=C/C2=CC3=C(C(=C2)O)O[C@@]4(C[C@H]([C@H](C([C@H]4C3)(C)C)O)O)C)O)/C)C",
    1847: "C1=CC=C(C=C1)C(=N)N",
    1849: "CC1=CC2=C(C(=C(C=C2C(=C1C3=C(C4=CC(=C(C(=C4C=C3C)C(=O)NC[C@H](C)C5=CC=CC=C5)O)O)O)O)O)O)C(=O)NC[C@H](C)C6=CC=CC=C6",
    1852: "C1CC2=C(C(=NN2C1)C3=CC=CC=N3)C4=C5C=CC(=CC5=NC=C4)OCCN6CCOCC6",
    1853: "CC1=C(C=CC(=C1)Br)S(=O)(=O)NC2=CC3=C(C=C2OC)N(C(=O)N3C)C",
    1854: "CC(C)C1=CC=C(C=C1)C2=CC(=O)C3=CC=CC=C3O2",
    1855: "C=CC(=O)N1CCN(CC1)C(=O)CNC2=CC(=C(C=C2O)Cl)I",
    1862: "CC(C)C[C@@H](C=O)NC(=O)[C@H](CC(C)C)NC(=O)[C@H](CC(C)C)NC(=O)OCC1=CC=CC=C1",
    1873: "C1COCCN1C2=NC(=NC(=C2)C3=CN=C(C=C3C(F)(F)F)N)N4CCOCC4",
    1908: "CC(C)NC1=NC=C(C(=C1)C2=CNC(=C2)C(=O)N[C@H](CO)C3=CC(=CC=C3)Cl)Cl",
    1909: "CC1(CCC(=C(C1)C2=CC=C(C=C2)Cl)CN3CCN(CC3)C4=CC(=C(C=C4)C(=O)NS(=O)(=O)C5=CC(=C(C=C5)NCC6CCOCC6)[N+](=O)[O-])OC7=CN=C8C(=C7)C=CN8)C",
    1910: "CN(C)CC[C@H](CSC1=CC=CC=C1)NC2=C(C=C(C=C2)S(=O)(=O)NC(=O)C3=CC=C(C=C3)N4CCN(CC4)CC5=CC=CC=C5C6=CC=C(C=C6)Cl)[N+](=O)[O-]",
    1911: "C[C@@H]1[C@@H](C(=O)N[C@@H](C(=O)N2CCC[C@H]2C(=O)N(CC(=O)N([C@H](C(=O)O1)C(C)C)C)C)C(C)C)NC(=O)C3=C4C(=C(C=C3)C)OC5=C(C(=O)C(=C(C5=N4)C(=O)N[C@H]6[C@H](OC(=O)[C@@H](N(C(=O)CN(C(=O)[C@@H]7CCCN7C(=O)[C@H](NC6=O)C(C)C)C)C)C(C)C)C)N)C",
    1912: "CN1C(=C(C=N1)Cl)C2=C(SC(=C2)C(=O)N[C@@H](CC3=CC(=CC=C3)F)CN)Cl",
    1913: "CC1=CC=CC=C1C(C(=O)NC2CCCCC2)N(C3=CC(=CC=C3)F)C(=O)CN4C=CN=C4C",
    1915: "C[C@@H]1CN(CCN1C(=O)OC2=C(C=C3C(=C2)C(=NC=N3)NC4=C(C(=CC=C4)Cl)F)OC)C",
    1916: "C1CN(CCC1(C(=O)N[C@@H](CCO)C2=CC=C(C=C2)Cl)N)C3=NC=NC4=C3C=CN4",
    1917: "C[C@@H]1COCCN1C2=NC(=NC(=C2)C3(CC3)[S@](=N)(=O)C)C4=CN=CC5=C4C=CN5",
    1918: "C[C@H](C1=CC(=CC2=C1OC(=CC2=O)N3CCOCC3)C(=O)N(C)C)NC4=CC(=CC(=C4)F)F",
    1919: "CN1C=C(C2=CC=CC=C21)C3=NC(=NC=C3)NC4=C(C=C(C(=C4)NC(=O)C=C)N(C)CCN(C)C)OC",
    1922: "CC1=CC2=C(N1)C=CC(=C2F)OC3=NC=NC4=CC(=C(C=C43)OC)OCCCN5CCCC5",
    1924: "C[C@@H]1C[C@H](C2=C1C(=NC=N2)N3CCN(CC3)C(=O)[C@H](CNC(C)C)C4=CC=C(C=C4)Cl)O",
    1925: "CC/C(=C(/C1=CC=C(C=C1)/C=C/C(=O)O)\\C2=CC3=C(C=C2)NN=C3)/C4=C(C=C(C=C4)F)Cl",
    1926: "CC1=C(SC2=C1N=C(N=C2N3CCOCC3)C4=CN=C(N=C4)N)C5(COC5)OC",
    1927: "C1=CC=C(C=C1)COC2=C(C=C(C=C2)C3=CC(=NC=C3)F)C(=O)NC4=CN=CC=C4",
    1928: "CCN1C=C(C2=C(C1=O)C=C(S2)C(=NC3CCS(=O)(=O)CC3)N)C4=CC(=CC=C4)C(F)(F)F",
    1930: "C1=CC(=CC(=C1)NC(=O)C2=C(C(=CC=C2)O)O)NC(=O)C3=C(C(=CC=C3)O)O",
    1931: "CCC(=O)OCN1C(=O)C=CC1=O",
    1932: "C1CCN(C1)CC2CC(C2)N3C=C(C4=C(N=CN=C43)N)C5=CC(=CC=C5)OCC6=CC=CC=C6",
    1933: "CC(=O)C1=CC(=C(S1)SC2=C(C=C(C=C2)F)F)[N+](=O)[O-]",
    1936: "C[C@@H](C1=CN2C=CN=C2C=C1)N3C4=NC(=CN=C4N=N3)C5=CN(N=C5)C",
    1939: "C1=CC=C2C(=C1)C(=CC(=C2O)SCC(=O)O)NS(=O)(=O)C3=CC=C(C=C3)Br",
    1940: "COC1=CC=C(C=C1)N2C(=NN=C2SCCCN3C(=O)C4=CC=CC5=C4C(=CC=C5)C3=O)C6=CC=NC=C6",
    1941: "CC1=[N+](C2=C(N1CCOC)C(=O)C3=CC=CC=C3C2=O)CC4=NC=CN=C4.[Br-]",
    1996: "CC1=CSC(=NC2CCCCC2)N1/N=C/C3=C(C(=C(C=C3)O)O)O",
    1997: "C1CC2=C(C=C(C=C2)C3=NC(=C(S3)CCCOC4=CC=C(C=C4)CN)C(=O)O)/C(=N/NC5=NC6=CC=CC=C6S5)/C1",
    2010: "CN1CCN(CC1)C2=NC3=CC(=C(C=C3C(=N2)NC4CCN(CC4)CC5=CC=CC=C5)OC)OC",
    2011: "COC1=C(C2=CC=CC=C2C=C1)CN3C=NC4=C(C3=O)C5=C(S4)CC(CC5)NCC6=CN=CC=C6",
    2037: "CCCC1=C(C(=O)NC(=C1)C)CNC(=O)C2=C3C=NN(C3=CC(=C2)C4=CC(=NC=C4)N5CCN(CC5)C)C(C)C",
    2038: "CC(C)N1CCC(CC1)NC2=NC(=NC3=CC(=C(C=C32)OC)OCCCN4CCCC4)C5CCCCC5",
    2039: "C1=CC2=C(C=CC=N2)C(=C1)NC(=O)/C(=C/C3=CC=C(O3)C4=C(C=CC(=C4)Cl)Cl)/C#N",
    2040: "COC1=CC2=C(C=CN=C2C=C1OCCCN3CCOCC3)OC4=C(C=C(C=C4)NC(=O)C5(CC5)C(=O)NC6=CC=C(C=C6)F)F",
    2043: "C/C(=C\\C(=O)NC1=CC=CC=C1C(=O)O)/C2=CC3=CC=CC=C3C=C2",
    2044: "C1=CC=C2C(=C1)C(=CC(=N2)NC(=O)C3=CC(=CC(=N3)C(=O)NC4=NC5=CC=CC=C5C(=C4)OCCN)OCCN)OCCN",
    2045: "C[C@@H](C1=C(N=C2C=C(C=CC2=C1)F)C3=CC=CC=N3)NC4=NC=NC5=C4NC=N5",
    2046: "CN1C=C(C=N1)C2=C3N=C(C(=C(N3N=C2)N)Br)[C@@H]4CCCNC4",
    2047: "CC(C)NC1=NC=C(C(=C1)C2=CNC(=C2)C(=O)N[C@H](CO)C3=CC(=CC=C3)Cl)Cl",
    2048: "CCC1=C[C@H]2C[C@@](C3=C(CN(C2)C1)C4=CC=CC=C4N3)(C5=C(C=C6C(=C5)[C@]78CCN9[C@H]7[C@@](C=CC9)([C@H]([C@@]([C@@H]8N6C)(C(=O)OC)O)OC(=O)C)CC)OC)C(=O)OC",
    2055: "CC1=C2COC(=O)C2=C(C(=C1OC)C/C=C(\\C)/CCC(=O)O)O",
    2057: "C1CCC(=NNC2=NC(=CS2)C3=CC=C(C=C3)C#N)C1",
    2096: "CC1=CN=C(N=C1C2=CNC(=C2)C(=O)N[C@H](CO)C3=CC(=CC=C3)Cl)NC4=C(C=C(C=C4)F)Cl",
    2106: "CN1C(=C(C=N1)Cl)C2=C(OC(=C2)C(=O)N[C@@H](CC3=CC(=C(C=C3)F)F)CN)Cl",
    2107: "C1COCCN1C2=CC=C(C=C2)C3=C(C=NC=C3)C4=CC(=C(C(=C4)F)O)F",
    2109: "C[C@@H]1CN(C[C@@H](N1)C)C2=NC=C(C(=C2)C)C3=CC=C(C=C3)C4=NC5=C(C=CN5C)C(=O)N4",
    2110: "C1CC(C1)NC2=NC=CC(=C2)C(=O)NC[C@@H](CN3CCC4=CC=CC=C4C3)O",
    2111: "CS(=O)(=O)C1=CC=C(C=C1)C2=CN=C(C(=N2)C(=O)NC3=CC=CC=C3)N",
    2148: "CC(C)(C)C(=O)OCOP(=O)(C1CCCN(C1=O)O)OCOC(=O)C(C)(C)C",
    2154: "C1CNCCC1N[C@@H]2C[C@H]2C3=CC=CC=C3",
    2156: "C1=NC(=NC(=O)N1[C@H]2[C@@H]([C@@H]([C@H](O2)CO)O)O)N",
    2157: "COC1=C(C=C2C(=C1)C3(CCC3)C(=N2)N)OCCCN4CCCC4",
    2158: "C[C@@H]1CC(=O)NC2=CC=CC(=C2N1)C3=CC4=C(C=C3)N(N=C4C5=CN(N=C5)C)C",
    2159: "COC1=C(C=C2C(=C1)C(=NC(=N2)N3CCCC3)NCCCCCN4CCCC4)OC",
    2169: "CC1=CN2C(=O)C=C(N=C2C(=C1)[C@@H](C)NC3=CC=CC=C3C(=O)O)N4CCOCC4",
    2170: "C1=CC(=CC=C1C2=CNN=C2)[C@@](CN)(C3=CC=C(C=C3)Cl)O",
    2171: "C[C@]1(CCCN1C2=NN3C=CC=C3C(=N2)NC4=NNC(=C4)C5CC5)C(=O)NC6=CN=C(C=C6)F",
    2172: "CC1=C(SC2=C1C(=N[C@H](C3=NN=C(N32)C)CC(=O)OC(C)(C)C)C4=CC=C(C=C4)Cl)C",
    2173: "CN1CC2=C(C=CC(=C2)NS(=O)(=O)C3=CC=CC=C3OC)NC1=O",
    2174: "C1=CC=C(C=C1)CN2C3=CC=CC=C3C(=C(C2=O)C(=O)NCC(=O)O)O",
    2175: "CC1=CN=C(N1)C2=CN=C(N=C2C3=C(C=C(C=C3)Cl)Cl)NCCNC4=NC=C(C=C4)C#N",
    2177: "CC(C)N(CCCNC(=O)NC1=CC=C(C=C1)C(C)(C)C)C[C@@H]2[C@H]([C@H]([C@@H](O2)N3C=C(C4=C(N=CN=C43)N)Br)O)O",
    2359: "CC1=C(C=C(C=N1)Cl)NCC2=CC=C(S2)C(=O)N[C@@H](CC3CCCC3)C(=O)NC4CC4",
    2438: "C([C@@H]([C@@H]1C(=C(C(=O)O1)O)O)O)O",
    2439: "C(CC(=O)N[C@@H](CS)C(=O)NCC(=O)O)[C@@H](C(=O)O)N",
    2498: "C1CSSC1CCCCC(=O)O",
    2499: "CC(=O)N[C@@H](CS)C(=O)O",
}
SMILES_PKL = PROC / 'drugs_with_smiles.parquet'
compounds  = pd.read_csv(DRUGS_CSV)

# ---- Priority 1: embedded seed dict ------------------------------------
_id_to_smi = {int(r['DRUG_ID']): _SMILES_SEED.get(int(r['DRUG_ID'])) for _, r in compounds.iterrows()}
_missing    = [r for _, r in compounds.iterrows() if _id_to_smi.get(int(r['DRUG_ID'])) is None]
print(f'Seed SMILES: {sum(v is not None for v in _id_to_smi.values())}/{len(compounds)} from embedded dict')
print(f'Need fetch:  {len(_missing)} drugs')

# ---- Priority 2: fetch missing from PubChem / ChEMBL -------------------
if _missing:
    _PUBCHEM_URL = ('https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{}'
                    '/property/CanonicalSMILES,IsomericSMILES/JSON')
    _CHEMBL_URL  = ('https://www.ebi.ac.uk/chembl/api/data/molecule.json'
                    '?pref_name__iexact={}&limit=1')
    _sess = requests.Session()
    _sess.headers['User-Agent'] = 'PathXDRP/1.0'

    def _get_pubchem(name):
        try:
            r = _sess.get(_PUBCHEM_URL.format(requests.utils.quote(name)), timeout=15)
            if r.status_code == 200:
                props = r.json()['PropertyTable']['Properties'][0]
                for k in ('IsomericSMILES', 'CanonicalSMILES'):
                    if props.get(k): return props[k]
        except Exception: pass
        return None

    def _get_chembl(name):
        try:
            r = _sess.get(_CHEMBL_URL.format(requests.utils.quote(name)), timeout=15)
            if r.status_code == 200:
                mols = r.json().get('molecules', [])
                if mols and mols[0].get('molecule_structures'):
                    return mols[0]['molecule_structures'].get('canonical_smiles')
        except Exception: pass
        return None

    n_fetched = 0
    for row in tqdm(_missing, desc='Fetching missing SMILES'):
        name = str(row['DRUG_NAME'])
        smi  = _get_pubchem(name) or _get_chembl(name)
        if smi is None and pd.notna(row.get('SYNONYMS')):
            syn = str(row['SYNONYMS']).split(',')[0].strip()
            smi = _get_pubchem(syn) or _get_chembl(syn)
        _id_to_smi[int(row['DRUG_ID'])] = smi
        if smi: n_fetched += 1
        time.sleep(0.22)
    print(f'Fetched {n_fetched}/{len(_missing)} additional SMILES')

# ---- build final dataframe ---------------------------------------------
rows = []
for _, row in compounds.iterrows():
    rows.append({
        'DRUG_ID':        row['DRUG_ID'],
        'DRUG_NAME':      row['DRUG_NAME'],
        'TARGET':         row.get('TARGET'),
        'TARGET_PATHWAY': row.get('TARGET_PATHWAY'),
        'SMILES':         _id_to_smi.get(int(row['DRUG_ID'])),
    })
df_s    = pd.DataFrame(rows)
n_found = df_s['SMILES'].notna().sum()
print(f'SMILES total: {n_found}/{len(df_s)} ({100*n_found/max(len(df_s),1):.1f}%)')
if n_found == 0:
    raise RuntimeError('0 SMILES available — cannot continue.')
df_s.to_parquet(SMILES_PKL, index=False)
print(f'Saved -> {SMILES_PKL.name}')


Seed SMILES: 502/621 from embedded dict
Need fetch:  119 drugs


Fetching missing SMILES:   0%|          | 0/119 [00:00<?, ?it/s]

Fetched 0/119 additional SMILES
SMILES total: 502/621 (80.8%)
Saved -> drugs_with_smiles.parquet


In [7]:
# -- 3.4  Load expression matrix ---------------------------------------------

def load_expression():
    mapping = pd.read_csv(COSMIC_MAP)
    cosmic_col = 'COSMICID' if 'COSMICID' in mapping.columns else 'COSMIC_ID'
    mapping = mapping[mapping[cosmic_col].notna()].copy()
    mapping[cosmic_col] = mapping[cosmic_col].astype(int)
    model_to_cosmic = dict(zip(mapping['ModelID'], mapping[cosmic_col]))
    print(f'  Mapping: {len(model_to_cosmic)} cell lines')

    print(f'  Loading expression matrix ({EXPR_FILE.stat().st_size/1024**2:.0f} MB) ...')
    t0 = time.time()
    expr = pd.read_csv(EXPR_FILE, index_col=0)
    print(f'    -> {expr.shape[0]} rows x {expr.shape[1]:,} cols in {time.time()-t0:.1f}s')
    # 26Q1 format: index=ProfileID, columns include is_default_entry + ModelID + genes
    if 'ModelID' in expr.columns:
        if 'is_default_entry' in expr.columns:
            expr = expr[expr['is_default_entry'] == True].copy()
        expr = expr.set_index('ModelID')
        _drop = [c for c in ('is_default_entry', 'ProfileID') if c in expr.columns]
        if _drop:
            expr = expr.drop(columns=_drop)
    # Drop any remaining non-numeric columns (safety net)
    expr = expr.select_dtypes(include='number')
    expr.columns = pd.Index([c.split(' (')[0].strip() for c in expr.columns])
    print(f'    -> {expr.shape[0]} models x {expr.shape[1]:,} genes after index normalisation')

    valid = [m for m in expr.index if m in model_to_cosmic]
    expr = expr.loc[valid].copy()
    expr.index = pd.Index([model_to_cosmic[m] for m in expr.index], name='COSMIC_ID', dtype=int)

    print('  Z-scoring ...')
    mu, sd = expr.mean(0), expr.std(0)
    sd[sd == 0] = 1.0
    expr = (expr - mu) / sd
    print(f'  Expression ready: {expr.shape}')
    return expr.astype('float32')

expr_matrix = load_expression()

  Mapping: 977 cell lines
  Loading expression matrix (291 MB) ...
    -> 1775 rows x 19,220 cols in 11.5s
    -> 1775 models x 19,215 genes after index normalisation
  Z-scoring ...
  Expression ready: (779, 19215)


In [8]:
# -- 3.5  Build master DataFrame ---------------------------------------------

def load_cell_metadata():
    cells = pd.read_excel(CELL_XLS)
    rename = {
        'COSMIC identifier': 'COSMIC_ID',
        'Sample Name': 'CELL_LINE_NAME',
        'GDSC\nTissue descriptor 1': 'tissue_1',
        'GDSC\nTissue\ndescriptor 2': 'tissue_2',
        'Cancer Type\n(matching TCGA label)': 'cancer_type',
        'Microsatellite \ninstability Status (MSI)': 'msi_status',
    }
    cells = cells.rename(columns={k: v for k, v in rename.items() if k in cells.columns})
    if 'COSMIC_ID' not in cells.columns:
        raise KeyError(
            f'COSMIC_ID column not found in Cell_Lines_Details.xlsx after renaming. '
            f'Columns present: {list(cells.columns)}')
    cells['COSMIC_ID'] = pd.to_numeric(cells['COSMIC_ID'], errors='coerce')
    return cells

# --- Step-by-step build with row counts at every filter ---
response = pd.read_csv(GDSC2_CSV)
print(f'[1] GDSC2 raw              : {len(response):>8,} rows')
response = response.drop_duplicates(subset=['DRUG_ID', 'COSMIC_ID'])
print(f'[2] after dedup            : {len(response):>8,} rows | '
      f'{response["DRUG_ID"].nunique()} drugs | {response["COSMIC_ID"].nunique()} cells')

drugs = pd.read_parquet(SMILES_PKL)
print(f'[3] drugs with SMILES      : {drugs["SMILES"].notna().sum():>8,} / {len(drugs)}')

cells = load_cell_metadata()
print(f'[4] cell metadata loaded   : {len(cells):>8,} rows')

df = response.merge(drugs[['DRUG_ID', 'SMILES', 'TARGET', 'TARGET_PATHWAY']], on='DRUG_ID', how='left')
print(f'[5] after drug merge       : {len(df):>8,} rows | SMILES missing: {df["SMILES"].isna().sum()}')

df = df.merge(cells[['COSMIC_ID', 'tissue_1', 'tissue_2', 'cancer_type', 'msi_status']],
              on='COSMIC_ID', how='left')
print(f'[6] after cell merge       : {len(df):>8,} rows')

df = df[df['SMILES'].notna()].copy()
print(f'[7] after SMILES filter    : {len(df):>8,} rows')

cells_with_expr = set(expr_matrix.index)
print(f'[8] expr_matrix cell IDs   : {len(cells_with_expr):>8,}')
df_cosmic = set(df['COSMIC_ID'].unique())
overlap = df_cosmic & cells_with_expr
print(f'[9] COSMIC overlap         : {len(overlap):>8,} cells '
      f'(response has {len(df_cosmic)}, expr has {len(cells_with_expr)})')

df = df[df['COSMIC_ID'].isin(cells_with_expr)].reset_index(drop=True)
print(f'[10] after expr filter     : {len(df):>8,} rows | '
      f'{df["DRUG_ID"].nunique()} drugs | {df["COSMIC_ID"].nunique()} cells')

if len(df) == 0:
    raise RuntimeError(
        'Master DataFrame is empty after all filters.\n'
        f'  GDSC2 cell IDs are integers (COSMIC_ID); expr_matrix index sample: '
        f'{list(expr_matrix.index[:5])}\n'
        '  Check that expr_matrix.index dtype matches GDSC2 COSMIC_ID dtype.\n'
        '  Also verify that cosmic_to_depmap.csv maps to cell lines present in both datasets.')

print(f'\nMaster DF: {len(df):,} rows | {df["DRUG_ID"].nunique()} drugs | {df["COSMIC_ID"].nunique()} cells')


[1] GDSC2 raw              :  242,036 rows
[2] after dedup            :  242,036 rows | 295 drugs | 969 cells
[3] drugs with SMILES      :      502 / 621


/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


[4] cell metadata loaded   :    1,002 rows
[5] after drug merge       :  242,036 rows | SMILES missing: 34686
[6] after cell merge       :  242,036 rows
[7] after SMILES filter    :  207,350 rows
[8] expr_matrix cell IDs   :      717
[9] COSMIC overlap         :      700 cells (response has 969, expr has 717)
[10] after expr filter     :  151,176 rows | 247 drugs | 700 cells

Master DF: 151,176 rows | 247 drugs | 700 cells


In [9]:
# -- 3.6  Build KEGG pathway -> gene map ------------------------------------

PGM_PATH = PROC / 'pathway_gene_map.json'

if PGM_PATH.exists():
    with open(PGM_PATH) as f: pathway_gene_symbols = json.load(f)
    print(f'pathway_gene_map.json: {len(pathway_gene_symbols)} pathways (cached)')
else:
    KEGG = 'https://rest.kegg.jp'
    DELAY = 0.4

    def _kegg_get(url, desc):
        time.sleep(DELAY)
        for attempt in range(3):
            try:
                r = requests.get(url, timeout=120, headers={'User-Agent':'pathxdrp/1.0'})
                r.raise_for_status(); return r.text
            except Exception as e:
                print(f'  [{desc}] attempt {attempt+1}/3: {e}')
                time.sleep(2**attempt)
        raise RuntimeError(f'KEGG fetch failed: {url}')

    print('Fetching KEGG pathway list ...')
    pw_text = _kegg_get(f'{KEGG}/list/pathway/hsa', 'pathway list')
    pathway_names = {}
    for line in pw_text.strip().split('\n'):
        if '\t' not in line: continue
        pid, name = line.split('\t', 1)
        pathway_names[pid.replace('path:', '')] = name.split(' - Homo sapiens')[0].strip()
    print(f'  {len(pathway_names)} human pathways')

    print('Fetching KEGG gene-pathway links (~30 s) ...')
    from collections import defaultdict
    gp_text = _kegg_get(f'{KEGG}/link/pathway/hsa', 'gene-pathway')
    pathway_entrez = defaultdict(list)
    for line in gp_text.strip().split('\n'):
        if '\t' not in line: continue
        gpart, ppart = line.strip().split('\t', 1)
        try: eid = int(gpart.replace('hsa:', ''))
        except ValueError: continue
        pathway_entrez[ppart.replace('path:', '')].append(eid)

    # Parse entrez->symbol from expression matrix column names ("TSPAN6 (7105)" format)
    print('Building entrez->symbol map from expression matrix header ...')
    hdr = pd.read_csv(EXPR_FILE, nrows=0, index_col=0)
    expr_gene_set = set(c.split(' (')[0].strip() for c in hdr.columns)
    entrez_to_sym = {}
    for col in hdr.columns:
        if ' (' in col and col.endswith(')'):
            sym = col.split(' (')[0].strip()
            try:
                eid = int(col.split('(')[1].rstrip(')').strip())
                if sym in expr_gene_set: entrez_to_sym[eid] = sym
            except ValueError: pass
    print(f'  {len(entrez_to_sym):,} entrez->symbol mappings')

    # Build pathway->[gene_symbols]
    pathway_gene_symbols = {}
    for pid, eids in pathway_entrez.items():
        if pid not in pathway_names: continue
        seen, syms = set(), []
        for eid in eids:
            s = entrez_to_sym.get(eid)
            if s and s not in seen: seen.add(s); syms.append(s)
        if syms: pathway_gene_symbols[pathway_names[pid]] = syms

    with open(PGM_PATH, 'w') as f: json.dump(pathway_gene_symbols, f)
    print(f'pathway_gene_map.json: {len(pathway_gene_symbols)} pathways saved')

Fetching KEGG pathway list ...
  370 human pathways
Fetching KEGG gene-pathway links (~30 s) ...
Building entrez->symbol map from expression matrix header ...
  19,215 entrez->symbol mappings
pathway_gene_map.json: 370 pathways saved


In [10]:
# -- 3.7  Build train/val/test splits --------------------------------------
import hashlib
from sklearn.model_selection import GroupKFold, KFold

SPLITS_DIR = PROC / 'splits'
N_FOLDS = 5
MIN_SAMPLES_PER_FOLD = 10   # absolute minimum to form meaningful val/test sets

# -- guard ------------------------------------------------------------------
if len(df) < N_FOLDS * MIN_SAMPLES_PER_FOLD:
    raise ValueError(
        f'Master DataFrame has only {len(df)} rows — need at least '
        f'{N_FOLDS * MIN_SAMPLES_PER_FOLD} to build {N_FOLDS}-fold splits.\n'
        'Re-run the master DF cell (3.5) and check the diagnostic output.')

def _df_hash(df):
    return hashlib.md5((str(sorted(df.columns.tolist())) + str(len(df))).encode()).hexdigest()[:8]

def _save_fold(d, tr, va, te):
    Path(d).mkdir(parents=True, exist_ok=True)
    for name, arr in [('train', tr), ('val', va), ('test', te)]:
        np.save(str(Path(d) / f'{name}.npy'), np.array(arr))

def _random_split(df, seed=0):
    folds = []
    for fold_i, (tr, te) in enumerate(
            KFold(N_FOLDS, shuffle=True, random_state=seed).split(np.arange(len(df)))):
        # Use a fold-specific RNG so val/test halves differ across folds
        rng = np.random.default_rng(seed * 1000 + fold_i)
        rng.shuffle(te)
        folds.append({'train': tr, 'val': te[:len(te)//2], 'test': te[len(te)//2:]})
    return folds

def _group_split(df, col, seed=0):
    groups = df[col].values
    n_unique = len(np.unique(groups))
    if n_unique < N_FOLDS:
        raise ValueError(
            f'Cannot build {N_FOLDS}-fold group split on column "{col}": '
            f'only {n_unique} unique groups present (need >= {N_FOLDS}).')
    folds = []
    for fold_i, (tr, te) in enumerate(
            GroupKFold(N_FOLDS).split(np.arange(len(df)), groups=groups)):
        rng = np.random.default_rng(seed * 1000 + fold_i)
        rng.shuffle(te)
        folds.append({'train': tr, 'val': te[:len(te)//2], 'test': te[len(te)//2:]})
    return folds

def _tissue_split(df, seed=0):
    if 'tissue_2' not in df.columns:
        raise ValueError(
            '_tissue_split: "tissue_2" column not found in df. '
            'Check that Cell_Lines_Details.xlsx was loaded and merged correctly.')
    tissues = df['tissue_2'].fillna('unknown').values
    top5 = df['tissue_2'].value_counts().index[:N_FOLDS].tolist()
    if len(top5) < N_FOLDS:
        raise ValueError(
            f'_tissue_split: only {len(top5)} tissue types with data, need {N_FOLDS}.')
    folds = []
    for fold_i, t in enumerate(top5):
        te = np.where(tissues == t)[0]
        tr = np.where(tissues != t)[0]
        rng = np.random.default_rng(seed * 1000 + fold_i)
        rng.shuffle(te)
        folds.append({'train': tr, 'val': te[:len(te)//2], 'test': te[len(te)//2:]})
    return folds

BUILDERS = {
    'random':       lambda df, s: _random_split(df, s),
    'cell_blind':   lambda df, s: _group_split(df, 'COSMIC_ID', s),
    'drug_blind':   lambda df, s: _group_split(df, 'DRUG_ID', s),
    'tissue_blind': lambda df, s: _tissue_split(df, s),
}

def load_split(name, seed, fold):
    d = SPLITS_DIR / name / f'seed{seed}' / f'fold{fold}'
    return np.load(d / 'train.npy'), np.load(d / 'val.npy'), np.load(d / 'test.npy')

dfhash = _df_hash(df)
for name, builder in BUILDERS.items():
    for seed in range(5):
        lock = SPLITS_DIR / name / f'seed{seed}' / 'meta.json'
        if lock.exists():
            saved = json.loads(lock.read_text())
            if saved.get('df_hash') == dfhash:
                continue
        folds = builder(df, seed)
        # Validate fold sizes before writing to disk
        for fold_i, fold in enumerate(folds):
            for part, arr in fold.items():
                if len(arr) == 0:
                    raise ValueError(
                        f'Split "{name}" seed={seed} fold={fold_i}: '
                        f'"{part}" set is empty. '
                        f'Dataset may be too small or groups too imbalanced.')
        for i, fold in enumerate(folds):
            _save_fold(SPLITS_DIR / name / f'seed{seed}' / f'fold{i}',
                       fold['train'], fold['val'], fold['test'])
        (SPLITS_DIR / name / f'seed{seed}').mkdir(parents=True, exist_ok=True)
        lock.write_text(json.dumps({'df_hash': dfhash, 'name': name, 'seed': seed}))

print('All splits built.')


All splits built.


---
## 4. Model Definitions (Inlined)

In [11]:
# -- 4.1  Molecular graph utilities -----------------------------------------
import warnings
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, rdFingerprintGenerator
from torch_geometric.data import Data, Batch

RDLogger.DisableLog('rdApp.*')
warnings.filterwarnings('ignore', message='.*torch-scatter.*')

ATOM_TYPES = ['C','N','O','S','F','Si','P','Cl','Br','Mg','Na','Ca','Fe','As','Al','I','B','V','K',
              'Tl','Yb','Sb','Sn','Ag','Pd','Co','Se','Ti','Zn','H','Li','Ge','Cu','Au','Ni','Cd',
              'In','Mn','Zr','Cr','Pt','Hg','Pb']
HYBRIDISATION = [Chem.rdchem.HybridizationType.SP, Chem.rdchem.HybridizationType.SP2,
                 Chem.rdchem.HybridizationType.SP3, Chem.rdchem.HybridizationType.SP3D,
                 Chem.rdchem.HybridizationType.SP3D2]
BOND_TYPES = [Chem.rdchem.BondType.SINGLE, Chem.rdchem.BondType.DOUBLE,
              Chem.rdchem.BondType.TRIPLE, Chem.rdchem.BondType.AROMATIC]
MORGAN_RADIUS, MORGAN_BITS = 2, 256
_MORGAN_GEN = rdFingerprintGenerator.GetMorganGenerator(radius=MORGAN_RADIUS, fpSize=MORGAN_BITS)

# Bond feature dimension: len(BOND_TYPES)+1 + 1 + 1 + 3 = 10
BOND_FEAT_DIM = len(BOND_TYPES) + 1 + 1 + 1 + 3  # bond_type_oh + conjugated + in_ring + stereo

def _one_hot(val, vocab):
    v = [0]*(len(vocab)+1); v[vocab.index(val) if val in vocab else len(vocab)] = 1; return v

def atom_features(atom, fg_vocab=None):
    sym = atom.GetSymbol()
    deg = [0]*11; deg[min(atom.GetDegree(),10)] = 1
    chg = [0]*7;  chg[min(max(atom.GetFormalCharge()+3,0),6)] = 1
    hyb = [0]*(len(HYBRIDISATION)+1)
    hyb[{h:i for i,h in enumerate(HYBRIDISATION)}.get(atom.GetHybridization(), len(HYBRIDISATION))] = 1
    chi = [0,0,0]
    ct = atom.GetChiralTag()
    if ct == Chem.rdchem.ChiralType.CHI_TETRAHEDRAL_CW: chi[0]=1
    elif ct == Chem.rdchem.ChiralType.CHI_TETRAHEDRAL_CCW: chi[1]=1
    else: chi[2]=1
    nhs = [0]*5; nhs[min(atom.GetTotalNumHs(),4)] = 1
    feats = _one_hot(sym,ATOM_TYPES) + deg + chg + hyb + [int(atom.GetIsAromatic())] + [int(atom.IsInRing())] + chi + nhs
    if fg_vocab is not None:
        fg_vec = [0]*len(fg_vocab)
        mol = atom.GetOwningMol(); ao = rdFingerprintGenerator.AdditionalOutput(); ao.AllocateBitInfoMap()
        _MORGAN_GEN.GetFingerprint(mol, additionalOutput=ao)
        bm = ao.GetBitInfoMap()
        for bit, origins in (bm.items() if bm else []):
            for center, radius in origins:
                if center == atom.GetIdx() and radius == MORGAN_RADIUS and bit in fg_vocab:
                    fg_vec[fg_vocab[bit]] = 1
        feats += fg_vec
    return feats

def bond_features(bond):
    bt = bond.GetBondType()
    bt_oh = [0]*(len(BOND_TYPES)+1); bt_oh[{b:i for i,b in enumerate(BOND_TYPES)}.get(bt,len(BOND_TYPES))] = 1
    stereo = bond.GetStereo()
    st = [0,0,0]
    if stereo == Chem.rdchem.BondStereo.STEREOE: st[0]=1
    elif stereo == Chem.rdchem.BondStereo.STEREOZ: st[1]=1
    else: st[2]=1
    return bt_oh + [int(bond.GetIsConjugated())] + [int(bond.IsInRing())] + st

def smiles_to_graph(smiles, fg_vocab=None, label=None):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None
    x = torch.tensor([atom_features(a, fg_vocab) for a in mol.GetAtoms()], dtype=torch.float)
    es, ed, ea = [], [], []
    for b in mol.GetBonds():
        i,j = b.GetBeginAtomIdx(), b.GetEndAtomIdx(); bf = bond_features(b)
        for u,v in [(i,j),(j,i)]: es.append(u); ed.append(v); ea.append(bf)
    if not es:
        ei = torch.zeros((2,0),dtype=torch.long)
        eattr = torch.zeros((0,BOND_FEAT_DIM),dtype=torch.float)  # 10-dim to match bond_features()
    else:
        ei = torch.tensor([es,ed],dtype=torch.long)
        eattr = torch.tensor(ea,dtype=torch.float)
    g = Data(x=x, edge_index=ei, edge_attr=eattr)
    if label is not None: g.y = torch.tensor([label],dtype=torch.float)
    return g

def build_fg_vocab(smiles_list, top_k=MORGAN_BITS):
    from collections import Counter
    cnt = Counter()
    gen = rdFingerprintGenerator.GetMorganGenerator(radius=MORGAN_RADIUS, fpSize=top_k)
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol is None: continue
        ao = rdFingerprintGenerator.AdditionalOutput(); ao.AllocateBitInfoMap()
        gen.GetFingerprint(mol, additionalOutput=ao)
        bm = ao.GetBitInfoMap()
        if bm: cnt.update(bm.keys())
    return {bit: idx for idx,(bit,_) in enumerate(cnt.most_common(top_k))}

print('Graph utilities loaded.')


Graph utilities loaded.


In [12]:
# -- 4.1b  Feature dimension audit ------------------------------------------
# Shows exactly which features are computed for atoms/bonds and what flows into the model head.

def _atom_feat_dim(fg_vocab_size=MORGAN_BITS):
    """Return the atom (node) feature dimension and a labelled breakdown."""
    parts = [
        ('atom type one-hot', len(ATOM_TYPES) + 1),   # 44 types + 1 unknown
        ('degree one-hot',    11),                      # 0..10
        ('formal charge',      7),                      # -3..+3
        ('hybridisation',     len(HYBRIDISATION) + 1), # 5 types + other
        ('is aromatic',        1),
        ('is in ring',         1),
        ('chirality',          3),                      # CW / CCW / none
        ('total Hs',           5),                      # 0..4
        ('FG Morgan bits',    fg_vocab_size),
    ]
    return parts

def _bond_feat_dim():
    """Return the bond (edge) feature dimension and a labelled breakdown."""
    parts = [
        ('bond type one-hot', len(BOND_TYPES) + 1),   # 4 types + other  → 5
        ('is conjugated',      1),
        ('is in ring',         1),
        ('stereo',             3),                      # E / Z / none
    ]
    return parts

def print_feature_audit(hidden_dim=256):
    atom_parts = _atom_feat_dim()
    bond_parts = _bond_feat_dim()
    node_in_dim = sum(d for _, d in atom_parts)
    edge_in_dim = sum(d for _, d in bond_parts)

    print('=' * 60)
    print('  FEATURE DIMENSION AUDIT')
    print('=' * 60)

    print('\n[Atom / Node features]  node_in_dim =', node_in_dim)
    for name, dim in atom_parts:
        print(f'    {name:<22s}  {dim:>4d}')
    print(f'    {"─"*28}')
    print(f'    {"TOTAL":22s}  {node_in_dim:>4d}')

    print('\n[Bond / Edge features]  edge_in_dim =', edge_in_dim)
    for name, dim in bond_parts:
        print(f'    {name:<22s}  {dim:>4d}')
    print(f'    {"─"*28}')
    print(f'    {"TOTAL":22s}  {edge_in_dim:>4d}')

    D = hidden_dim
    print(f'\n[Model head input]  (hidden_dim D = {D})')
    head_parts = [
        ('h_drug_ctx',    D, 'cross-attended drug context (global_add_pool × attn)'),
        ('h_mol (mean)',  D, 'drug graph global mean pool'),
        ('h_mol (max)',   D, 'drug graph global max pool  ┘ concatenated as h_mol'),
        ('h_cell_g',      D, 'pathway-attentive cell pool (learned query)'),
        ('interaction',   D, 'h_drug_ctx * h_cell_g  (element-wise)'),
    ]
    total_head = 0
    for name, dim, note in head_parts:
        print(f'    {name:<18s}  {dim:>4d}   # {note}')
        total_head += dim
    print(f'    {"─"*28}')
    print(f'    {"head in_dim":18s}  {total_head:>4d}   = 5 × D')

    print('\n[Consistency check]')
    assert edge_in_dim == BOND_FEAT_DIM, \
        f'BOND_FEAT_DIM ({BOND_FEAT_DIM}) mismatch with bond_features() count ({edge_in_dim})'
    print(f'  BOND_FEAT_DIM constant = {BOND_FEAT_DIM}  ✓')
    print(f'  head in_dim = 5 × {D} = {5*D}  ✓')
    print('=' * 60)

print_feature_audit(hidden_dim=256)


  FEATURE DIMENSION AUDIT

[Atom / Node features]  node_in_dim = 334
    atom type one-hot         44
    degree one-hot            11
    formal charge              7
    hybridisation              6
    is aromatic                1
    is in ring                 1
    chirality                  3
    total Hs                   5
    FG Morgan bits           256
    ────────────────────────────
    TOTAL                    334

[Bond / Edge features]  edge_in_dim = 10
    bond type one-hot          5
    is conjugated              1
    is in ring                 1
    stereo                     3
    ────────────────────────────
    TOTAL                     10

[Model head input]  (hidden_dim D = 256)
    h_drug_ctx           256   # cross-attended drug context (global_add_pool × attn)
    h_mol (mean)         256   # drug graph global mean pool
    h_mol (max)          256   # drug graph global max pool  ┘ concatenated as h_mol
    h_cell_g             256   # pathway-attentive cel

In [13]:
# -- 4.2  Drug GAT encoder --------------------------------------------------
from torch_geometric.nn import GATv2Conv, global_mean_pool, global_max_pool

class DrugGATEncoder(nn.Module):
    def __init__(self, node_in_dim, edge_in_dim, hidden_dim=128, n_layers=3,
                 n_heads=8, dropout=0.1, use_molformer=False):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.node_proj = nn.Linear(node_in_dim, hidden_dim)
        self.convs = nn.ModuleList([
            GATv2Conv(hidden_dim, hidden_dim//n_heads, heads=n_heads,
                      edge_dim=edge_in_dim, concat=True, dropout=dropout)
            for _ in range(n_layers)])
        self.norms = nn.ModuleList([nn.LayerNorm(hidden_dim) for _ in range(n_layers)])
        self.dropout = nn.Dropout(dropout)
        self.act = nn.GELU()

    def forward(self, data, smiles_list=None, batch=None):
        x = self.act(self.node_proj(data.x))
        for conv, norm in zip(self.convs, self.norms):
            x = norm(x + self.dropout(conv(x, data.edge_index, data.edge_attr)))
            x = self.act(x)
        if batch is None:
            batch = data.batch if hasattr(data,'batch') and data.batch is not None \
                    else torch.zeros(x.size(0), dtype=torch.long, device=x.device)
        # h_mol: (B, 2*hidden_dim)  -  mean+max concat for richer global readout
        h_mol = torch.cat([global_mean_pool(x, batch), global_max_pool(x, batch)], dim=-1)
        return x, h_mol  # (N_atoms, D), (B, 2D)

print('DrugGATEncoder defined.')

DrugGATEncoder defined.


In [14]:
# -- 4.3  Graph-Mamba drug encoder (requires mamba_ssm) ---------------------
from torch_geometric.utils import degree, to_dense_batch

try:
    from mamba_ssm import Mamba as _MambaCls
    MAMBA_PKG = True
except Exception:
    _MambaCls = None
    MAMBA_PKG = False

class _BiMambaBlock(nn.Module):
    def __init__(self, hidden_dim, d_state=16, d_conv=4):
        super().__init__()
        if not MAMBA_PKG: raise ImportError('mamba_ssm not installed')
        self.fwd  = _MambaCls(d_model=hidden_dim, d_state=d_state, d_conv=d_conv)
        self.bwd  = _MambaCls(d_model=hidden_dim, d_state=d_state, d_conv=d_conv)
        self.gate = nn.Linear(hidden_dim*2, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)

    def forward(self, x, pad_mask):
        x_in = x * pad_mask.unsqueeze(-1).float()
        h_f = self.fwd(x_in)
        h_b = torch.flip(self.bwd(torch.flip(x_in, dims=[1])), dims=[1])
        return self.norm(x + self.gate(torch.cat([h_f, h_b], dim=-1)))

class GraphMambaDrugEncoder(nn.Module):
    def __init__(self, node_in_dim, edge_in_dim, hidden_dim=128,
                 n_gat_layers=2, n_mamba_layers=2, n_heads=8, dropout=0.1, ordering='degree'):
        super().__init__()
        if not MAMBA_PKG: raise ImportError('mamba_ssm not installed')
        self.hidden_dim = hidden_dim
        self.ordering = ordering
        self.node_proj = nn.Linear(node_in_dim, hidden_dim)
        self.gat_convs = nn.ModuleList([
            GATv2Conv(hidden_dim, hidden_dim//n_heads, heads=n_heads,
                      edge_dim=edge_in_dim, concat=True, dropout=dropout)
            for _ in range(n_gat_layers)])
        self.gat_norms = nn.ModuleList([nn.LayerNorm(hidden_dim) for _ in range(n_gat_layers)])
        self.mamba_blocks = nn.ModuleList([_BiMambaBlock(hidden_dim) for _ in range(n_mamba_layers)])
        self.dropout = nn.Dropout(dropout)
        self.act = nn.GELU()

    def _atom_order(self, edge_index, batch_idx):
        n = batch_idx.numel()
        if self.ordering == 'canonical': return torch.arange(n, device=batch_idx.device)
        deg = degree(edge_index[0], num_nodes=n).to(batch_idx.device)
        order = torch.empty(n, dtype=torch.long, device=batch_idx.device)
        for b in batch_idx.unique():
            mask = batch_idx == b; idx = torch.nonzero(mask, as_tuple=False).squeeze(-1)
            order[mask] = idx[torch.argsort(deg[idx], descending=True)]
        return order

    def forward(self, data, smiles_list=None, batch=None):
        if batch is None:
            batch = data.batch if hasattr(data,'batch') and data.batch is not None \
                    else torch.zeros(data.x.size(0), dtype=torch.long, device=data.x.device)
        x = self.act(self.node_proj(data.x))
        for conv, norm in zip(self.gat_convs, self.gat_norms):
            x = norm(x + self.dropout(conv(x, data.edge_index, data.edge_attr)))
            x = self.act(x)
        perm = self._atom_order(data.edge_index, batch)
        inv_perm = torch.empty_like(perm); inv_perm[perm] = torch.arange(perm.numel(), device=perm.device)
        x_pad, pad_m = to_dense_batch(x[perm], batch[perm])
        for blk in self.mamba_blocks: x_pad = blk(x_pad, pad_m)
        x_perm_out = x_pad[pad_m]
        x_out = torch.empty_like(x_perm_out); x_out[perm] = x_perm_out
        # Return (B, 2D) h_mol to match DrugGATEncoder interface
        h_mol = torch.cat([global_mean_pool(x_out, batch), global_max_pool(x_out, batch)], dim=-1)
        return x_out, h_mol

print(f'GraphMambaDrugEncoder defined (MAMBA_PKG={MAMBA_PKG}).')

GraphMambaDrugEncoder defined (MAMBA_PKG=True).


In [15]:
# -- 4.4  PathwaySet cell encoder -------------------------------------------

class PathwaySetEncoder(nn.Module):
    def __init__(self, n_genes, pathway_gene_map, hidden_dim=128,
                 dropout=0.1, n_pw_transformer_layers=1):
        super().__init__()
        self.pathway_names = sorted(pathway_gene_map.keys())
        self.n_pathways = len(self.pathway_names)
        self.hidden_dim = hidden_dim
        self.gene_proj = nn.Sequential(nn.Linear(3, hidden_dim), nn.GELU(), nn.LayerNorm(hidden_dim))
        _nh = max(1, hidden_dim//32)
        self.pathway_transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=hidden_dim, nhead=_nh,
                dim_feedforward=hidden_dim*2, dropout=0.0,
                batch_first=True, norm_first=True),
            num_layers=n_pw_transformer_layers, enable_nested_tensor=False
        ) if n_pw_transformer_layers > 0 else None
        self.pathway_norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)

        # Pre-compute averaging matrix
        flat_gene, flat_pw = [], []
        for p_idx, pname in enumerate(self.pathway_names):
            for g in pathway_gene_map[pname]:
                flat_gene.append(g); flat_pw.append(p_idx)
        pw_sizes = torch.tensor([float(max(len(pathway_gene_map[p]),1)) for p in self.pathway_names])
        avg_mat = torch.zeros(self.n_pathways, max(len(flat_gene),1))
        for i,(pw,_) in enumerate(zip(flat_pw, flat_gene)):
            avg_mat[pw, i] = 1.0 / pw_sizes[pw].item()
        self.register_buffer('avg_matrix', avg_mat)
        self.register_buffer('flat_gene_idx',
            torch.tensor(flat_gene, dtype=torch.long) if flat_gene else torch.zeros(0, dtype=torch.long))
        self._n_pairs = len(flat_gene)

    def forward(self, expr):
        B = expr.size(0)
        if self._n_pairs == 0:
            return self.dropout(torch.zeros(B, self.n_pathways, self.hidden_dim, device=expr.device))
        # Force float32 for the statistical ops: fp16 catastrophic cancellation in (E[x^2]-E[x]^2)
        with torch.amp.autocast(device_type=expr.device.type, enabled=False):
            ge      = expr.float()[:, self.flat_gene_idx]
            avg     = self.avg_matrix.float()
            mean_pw = torch.matmul(ge,           avg.T)
            std_pw  = (torch.matmul(ge**2,       avg.T) - mean_pw**2).clamp(min=0).sqrt()
            frac_pw = torch.matmul((ge > 0).float(), avg.T)
        h = self.gene_proj(torch.stack([mean_pw, std_pw, frac_pw], dim=-1))
        if self.pathway_transformer is not None: h = self.pathway_transformer(h)
        return self.dropout(self.pathway_norm(h))

print('PathwaySetEncoder defined.')

PathwaySetEncoder defined.


In [16]:
# -- 4.5  GeneMamba cell encoder (requires mamba_ssm + HuggingFace transformers) ---------
from transformers import AutoModel, AutoTokenizer

class GeneMambaCellEncoder(nn.Module):
    def __init__(self, n_genes, gene_symbols, pathway_gene_map, hidden_dim=128,
                 top_k=2048, backbone_id='mineself2016/GeneMamba', freeze_backbone=True):
        super().__init__()
        if not MAMBA_PKG:
            raise ImportError('mamba_ssm is required for GeneMambaCellEncoder but is not installed.')
        self.top_k = min(top_k, n_genes)
        self.gene_symbols = list(gene_symbols)
        self.pathway_names = sorted(pathway_gene_map.keys())
        self.n_pathways = len(self.pathway_names)
        self.pathway_gene_map = pathway_gene_map
        self.n_genes = n_genes
        self.hidden_dim = hidden_dim

        # Load HF backbone — no fallback; raise immediately on failure
        print(f'[GeneMamba] Loading backbone from {backbone_id} ...')
        self.backbone = AutoModel.from_pretrained(backbone_id, trust_remote_code=True)
        self.backbone_dim = (
            getattr(self.backbone.config, 'hidden_size', None)
            or getattr(self.backbone.config, 'd_model', None)
        )
        if self.backbone_dim is None:
            raise ValueError(
                f'[GeneMamba] Cannot determine hidden_size/d_model from backbone config: '
                f'{self.backbone.config}')
        self.backbone_dim = int(self.backbone_dim)
        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad_(False)
        print(f'[GeneMamba] backbone_dim={self.backbone_dim}, frozen={freeze_backbone}')

        # Build gene → token ID map from the backbone's own tokenizer
        print(f'[GeneMamba] Building gene token map ...')
        tok = AutoTokenizer.from_pretrained(backbone_id, trust_remote_code=True)
        vocab = tok.get_vocab()
        ids = [vocab.get(g, vocab.get(g.upper(), -1)) for g in self.gene_symbols]
        n_missing = sum(1 for i in ids if i < 0)
        if n_missing == n_genes:
            raise ValueError(
                f'[GeneMamba] None of the {n_genes} gene symbols matched the backbone vocab '
                f'(checked both original case and upper). Check that backbone_id is correct.')
        if n_missing > 0:
            print(f'[GeneMamba] WARNING: {n_missing}/{n_genes} gene symbols not in vocab (will be skipped in top-k).')
        self.register_buffer('gene_token_ids',
                             torch.tensor(ids, dtype=torch.long))
        print(f'[GeneMamba] {n_genes - n_missing}/{n_genes} genes mapped to backbone vocab.')

        self.gene_adapter = nn.Sequential(
            nn.Linear(self.backbone_dim, hidden_dim), nn.GELU(), nn.LayerNorm(hidden_dim))
        self.pathway_norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(0.1)

    def forward(self, expr):
        _, topk_idx = expr.topk(self.top_k, dim=1)
        gene_emb = self._encode_topk(topk_idx)
        gene_emb = self.gene_adapter(gene_emb)
        return self.dropout(self._pool_pathways(gene_emb, topk_idx, expr.device))

    def _encode_topk(self, topk_idx):
        # Mask out genes with no vocab entry (id == -1) by clamping to 0;
        # their embeddings will be zeroed out after pooling via the pw_mask.
        token_ids = self.gene_token_ids[topk_idx].clamp(min=0)
        with torch.no_grad():
            out = self.backbone(input_ids=token_ids)
        h = getattr(out, 'last_hidden_state', None)
        if h is None:
            raise AttributeError(
                '[GeneMamba] backbone output has no last_hidden_state attribute. '
                'Verify the backbone architecture is a sequence model.')
        return h

    def _pool_pathways(self, gene_emb, topk_idx, device):
        B, K, D = gene_emb.shape
        if not hasattr(self, '_pw_mask'):
            mask = torch.zeros(self.n_pathways, self.n_genes, dtype=torch.bool)
            for p_i, pn in enumerate(self.pathway_names):
                for g in self.pathway_gene_map[pn]:
                    mask[p_i, g] = True
            self.register_buffer('_pw_mask', mask, persistent=False)
        # Zero out positions where gene_token_ids == -1 (unmapped genes)
        valid_mask = (self.gene_token_ids[topk_idx] >= 0).unsqueeze(1).expand(B, self.n_pathways, K)
        belong = self._pw_mask.to(device)[:, topk_idx].permute(1, 0, 2).float()  # (B,P,K)
        belong = belong * valid_mask.float()
        counts = belong.sum(-1, keepdim=True).clamp(min=1.0)
        out = torch.einsum('bpk,bkd->bpd', belong, gene_emb) / counts
        return self.pathway_norm(out)

print('GeneMambaCellEncoder defined (hard dependency: transformers + mamba_ssm).')


GeneMambaCellEncoder defined (hard dependency: transformers + mamba_ssm).


In [17]:
# -- 4.6  Cross-attention + Evidential head + PathXDRP ---------------------
import math
from torch_geometric.nn import global_add_pool

class PathwayMaskedCrossAttention(nn.Module):
    def __init__(self, hidden_dim=128, n_heads=8, dropout=0.1,
                 mask_type='soft', entropy_reg_weight=0.01, n_pathways=0):
        super().__init__()
        assert hidden_dim % n_heads == 0
        self.n_heads = n_heads; self.head_dim = hidden_dim//n_heads
        self.scale = math.sqrt(self.head_dim)
        self.mask_type = mask_type; self.entropy_reg_weight = entropy_reg_weight
        self.hidden_dim = hidden_dim
        self.q_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.k_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.v_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.out_proj = nn.Linear(hidden_dim, hidden_dim)
        self.norm_q  = nn.LayerNorm(hidden_dim); self.norm_kv = nn.LayerNorm(hidden_dim)
        self.attn_drop = nn.Dropout(dropout)
        if mask_type == 'soft' and n_pathways > 0:
            self.soft_mask_logit = nn.Parameter(torch.zeros(1,1,1,n_pathways))
        self._last_attn = None; self._entropy_loss = torch.tensor(0.0)

    def forward(self, h_drug, h_cell, atom_batch, hard_mask=None):
        B, N_pw = h_cell.size(0), h_cell.size(1)
        Q = self.q_proj(self.norm_q(h_drug))
        K = self.k_proj(self.norm_kv(h_cell)); V = self.v_proj(h_cell)
        Q_pad, pad_mask = to_dense_batch(Q, atom_batch)
        max_n = Q_pad.size(1)
        def to_mh(x, seq): return x.view(x.size(0),seq,self.n_heads,self.head_dim).permute(0,2,1,3)
        Q_mh = to_mh(Q_pad, max_n); K_mh = to_mh(K, N_pw); V_mh = to_mh(V, N_pw)
        scores = torch.einsum('bhnd,bhpd->bhnp', Q_mh, K_mh) / self.scale
        scores = scores.masked_fill((~pad_mask).unsqueeze(1).unsqueeze(-1), -1e4)
        if self.mask_type == 'soft' and hasattr(self, 'soft_mask_logit'):
            scores = scores + self.soft_mask_logit
        attn = self.attn_drop(F.softmax(scores, dim=-1))
        ctx = self.out_proj(
            (torch.einsum('bhnp,bhpd->bhnd', attn, V_mh)
             .permute(0,2,1,3).contiguous().view(B, max_n, self.hidden_dim))
        )[pad_mask]
        attn_flat = attn.mean(1)[pad_mask]
        p = attn_flat + 1e-9
        self._entropy_loss = -(p * p.log()).sum(-1).mean()
        self._last_attn = attn_flat.detach()
        return ctx, attn_flat


class EvidentialRegressionHead(nn.Module):
    def __init__(self, in_dim, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.GELU(), nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, 4))
    def forward(self, z):
        out = self.net(z)
        with torch.amp.autocast(device_type=z.device.type, enabled=False):
            out = out.float()
            out = out.clamp(-20, 20)  # prevent extreme raw outputs
            gamma = out[:,0]; nu = F.softplus(out[:,1]).clamp(min=0.05)+0.05
            alpha = F.softplus(out[:,2]).clamp(min=0.01)+1.1
            beta  = F.softplus(out[:,3]).clamp(min=1e-4)+1e-4
            aleat = (beta/(alpha-1)).clamp(max=1e4)
            epist = (beta/(nu*(alpha-1))).clamp(max=1e4)
        return {'mu':gamma,'nu':nu,'alpha':alpha,'beta':beta,'pred':gamma,'aleatoric':aleat,'epistemic':epist}

def evidential_loss(pred, y, lam=0.1):
    with torch.amp.autocast(device_type=y.device.type, enabled=False):
        g,nu,al,be,yf = pred['mu'].float(),pred['nu'].float(),pred['alpha'].float(),pred['beta'].float(),y.float()
        tbl = 2*be*(1+nu)
        nu_s = nu.clamp(min=1e-6); tbl_s = tbl.clamp(min=1e-6)
        nll = (0.5*torch.log(torch.tensor(torch.pi,device=y.device)/nu_s)
               - al*torch.log(tbl_s) + (al+0.5)*torch.log((nu_s*(yf-g)**2+tbl_s).clamp(min=1e-6))
               + torch.lgamma(al) - torch.lgamma(al+0.5))
        reg = torch.abs(yf-g) * (2*nu+al)
    return (nll + lam*reg).mean()


class PathXDRP(nn.Module):
    def __init__(self, node_in_dim, edge_in_dim, n_genes, pathway_gene_map,
                 hidden_dim=128, n_gat_layers=3, n_attn_heads=8, dropout=0.1,
                 mask_type='soft', entropy_reg_weight=0.01, evidential_lam=0.01,
                 drug_encoder_type='gat', cell_encoder_type='pathway_set',
                 gene_symbols=None, graph_mamba_kwargs=None, gene_mamba_kwargs=None,
                 n_pw_transformer_layers=1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.evidential_lam = evidential_lam
        self.entropy_reg_weight = entropy_reg_weight
        self.drug_encoder_type = drug_encoder_type

        if drug_encoder_type == 'graph_mamba':
            gm = dict(graph_mamba_kwargs or {})
            self.drug_enc = GraphMambaDrugEncoder(
                node_in_dim, edge_in_dim, hidden_dim, n_heads=n_attn_heads, dropout=dropout, **gm)
        else:
            self.drug_enc = DrugGATEncoder(
                node_in_dim, edge_in_dim, hidden_dim, n_gat_layers, n_attn_heads, dropout)

        if cell_encoder_type == 'gene_mamba':
            gm2 = dict(gene_mamba_kwargs or {})
            self.cell_enc = GeneMambaCellEncoder(
                n_genes, gene_symbols, pathway_gene_map, hidden_dim, **gm2)
        else:
            self.cell_enc = PathwaySetEncoder(
                n_genes, pathway_gene_map, hidden_dim, dropout, n_pw_transformer_layers)

        self.cross_attn = PathwayMaskedCrossAttention(
            hidden_dim, n_attn_heads, dropout, mask_type, entropy_reg_weight, len(pathway_gene_map))
        self.pool_proj = nn.Linear(hidden_dim, hidden_dim)
        self.pool_norm = nn.LayerNorm(hidden_dim)
        self.cell_pool_q = nn.Parameter(torch.randn(1,1,hidden_dim)*0.02)
        # in_dim = D (drug_context) + 2D (h_mol from mean+max pool) + D (cell_global) + D (interaction) = 5D
        self.head = EvidentialRegressionHead(in_dim=hidden_dim*5, hidden_dim=hidden_dim)

    @staticmethod
    def _assert_no_nan(t, name):
        if torch.isnan(t).any() or torch.isinf(t).any():
            raise RuntimeError(
                f"{name} contains NaN/Inf: shape={tuple(t.shape)} "
                f"nan={torch.isnan(t).sum().item()} inf={torch.isinf(t).sum().item()}")

    def forward(self, drug_batch, expr, smiles_list=None, hard_mask=None, y=None):
        h_atom, h_mol = self.drug_enc(drug_batch, smiles_list=smiles_list, batch=drug_batch.batch)
        self._assert_no_nan(h_mol,     "h_mol (drug encoder)")
        h_cell = self.cell_enc(expr)
        self._assert_no_nan(h_cell,    "h_cell (cell encoder)")
        context, attn_w = self.cross_attn(h_atom, h_cell, drug_batch.batch, hard_mask)
        self._assert_no_nan(context,   "context (cross-attn)")
        a_w = attn_w.max(-1, keepdim=True)[0]
        h_drug_ctx = self.pool_norm(self.pool_proj(global_add_pool(context*a_w, drug_batch.batch)))
        self._assert_no_nan(h_drug_ctx,"h_drug_ctx (pool_proj+norm)")
        pool_s = torch.matmul(
            self.cell_pool_q.expand(h_cell.size(0),-1,-1), h_cell.transpose(-1,-2)
        ) / math.sqrt(self.hidden_dim)
        h_cell_g = (F.softmax(pool_s, dim=-1) @ h_cell).squeeze(1)
        self._assert_no_nan(h_cell_g,  "h_cell_g (cell pool)")
        z = torch.cat([h_drug_ctx, h_mol, h_cell_g, h_drug_ctx*h_cell_g], dim=-1)
        pred = self.head(z)
        out = {'pred': pred, 'attn_weights': attn_w}
        if y is not None:
            ml = evidential_loss(pred, y, lam=self.evidential_lam)
            el = self.cross_attn._entropy_loss * self.entropy_reg_weight
            out['loss'] = ml + el; out['main_loss'] = ml; out['entropy_loss'] = el
        return out

print('PathXDRP model defined.')

PathXDRP model defined.


In [18]:
# -- 4.7  Metrics ------------------------------------------------------------
from scipy import stats
from sklearn.metrics import mean_squared_error, r2_score

def regression_report(y_true, y_pred, drug_ids=None, cell_ids=None, uncertainties=None):
    # Guard against NaN predictions (numerical instability during training)
    valid = np.isfinite(y_pred) & np.isfinite(y_true)
    nan_frac = (~valid).mean()
    if nan_frac > 0:
        print(f'  WARNING: {(~valid).sum()} / {len(valid)} non-finite predictions ')
        y_pred = y_pred[valid]; y_true = y_true[valid]
        if drug_ids is not None: drug_ids = drug_ids[valid]
        if cell_ids is not None: cell_ids = cell_ids[valid]
        if uncertainties is not None: uncertainties = uncertainties[valid]
    if len(y_pred) < 2:
        return {'RMSE': float('nan'), 'MAE': float('nan'), 'PCC': float('nan'),
                'Spearman': float('nan'), 'R2': float('nan')}
    out = {
        'RMSE':     float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'MAE':      float(np.mean(np.abs(y_true-y_pred))),
        'PCC':      float(stats.pearsonr(y_true, y_pred)[0]),
        'Spearman': float(stats.spearmanr(y_true, y_pred)[0]),
        'R2':       float(r2_score(y_true, y_pred)),
    }
    if drug_ids is not None:
        rs = []
        for d in np.unique(drug_ids):
            mask = drug_ids == d
            if mask.sum() < 2: continue
            yt, yp = y_true[mask], y_pred[mask]
            if np.std(yp) < 1e-8 or np.std(yt) < 1e-8: continue
            try:
                r = float(stats.pearsonr(yt, yp)[0])
                if np.isfinite(r): rs.append(r)
            except Exception: pass
        out['Per-drug PCC'] = float(np.mean(rs)) if rs else float('nan')
    if uncertainties is not None and len(uncertainties)==len(y_true):
        order = np.argsort(uncertainties)
        bins = np.array_split(order, 15)
        ece = sum(abs(np.sqrt(np.mean((y_true[b]-y_pred[b])**2)) - np.sqrt(np.mean(uncertainties[b])))
                  * len(b)/len(y_true) for b in bins if len(b))
        out['ECE'] = float(ece)
    return out

print('Metrics defined.')


Metrics defined.


---
## 5. Dataset & Training Loop

In [19]:
# -- 5.1  Dataset and DataLoader ---------------------------------------------
import random
from datetime import timedelta
from torch.utils.data import Dataset, DataLoader

class GDSCDataset(Dataset):
    def __init__(self, df, graph_cache, expr_matrix):
        self.df = df.reset_index(drop=True)
        self.gc = graph_cache
        self.expr_np = expr_matrix.values.astype('float32')
        self.cid2row = {int(cid): i for i, cid in enumerate(expr_matrix.index)}
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return {'drug_graph': self.gc[int(row['DRUG_ID'])],
                'expr':       self.expr_np[self.cid2row[int(row['COSMIC_ID'])]],
                'y':          float(row['LN_IC50']),
                'drug_id':    int(row['DRUG_ID']),
                'cosmic_id':  int(row['COSMIC_ID'])}

def collate_fn(batch):
    return {
        'drug_batch': Batch.from_data_list([b['drug_graph'] for b in batch]),
        'expr':  torch.tensor(np.stack([b['expr'] for b in batch]), dtype=torch.float),
        'y':     torch.tensor([b['y'] for b in batch], dtype=torch.float),
        'drug_ids':   torch.tensor([b['drug_id'] for b in batch], dtype=torch.long),
        'cosmic_ids': torch.tensor([b['cosmic_id'] for b in batch], dtype=torch.long),
    }

def build_graph_cache(drugs_df):
    smiles_list = drugs_df['SMILES'].dropna().tolist()
    print(f'  Building FG vocab from {len(smiles_list)} SMILES ...')
    fg_vocab = build_fg_vocab(smiles_list)
    cache, failed = {}, 0
    for _, row in tqdm(drugs_df.iterrows(), total=len(drugs_df), desc='  Drug graphs'):
        smi = row['SMILES']
        if pd.isna(smi): failed += 1; continue
        g = smiles_to_graph(smi, fg_vocab=fg_vocab)
        if g: cache[int(row['DRUG_ID'])] = g
        else: failed += 1
    print(f'  Graph cache: {len(cache)} drugs ({failed} failed)')
    return cache, fg_vocab

print('Dataset classes defined.')

Dataset classes defined.


In [20]:
# -- 5.2  Training and evaluation functions ----------------------------------

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def _nan_diagnostics(out, y):
    """Return a detailed diagnostic string for a batch that produced NaN/Inf loss."""
    lines = [f'  y        : min={y.min():.4f}  max={y.max():.4f}  mean={y.mean():.4f}']
    pred = out.get('pred', {})
    for key in ('mu', 'nu', 'alpha', 'beta', 'aleatoric', 'epistemic'):
        v = pred.get(key)
        if v is not None:
            has_nan = torch.isnan(v).any().item()
            has_inf = torch.isinf(v).any().item()
            lines.append(
                f'  pred.{key:<9s}: min={v.min():.4f}  max={v.max():.4f}'
                f'  nan={has_nan}  inf={has_inf}')
    for lk in ('loss', 'main_loss', 'entropy_loss'):
        v = out.get(lk)
        if v is not None:
            lines.append(f'  {lk:<14s}: {v.item():.6f}')
    return '\n'.join(lines)

def train_one_epoch(model, loader, optimizer, device, scaler=None, grad_clip=1.0):
    model.train(); total_loss = 0.0; n = 0
    use_amp = scaler is not None and device.type == 'cuda'
    for batch in tqdm(loader, desc='  train', leave=False):
        optimizer.zero_grad()
        db   = batch['drug_batch'].to(device)
        expr = batch['expr'].to(device)
        y    = batch['y'].to(device)
        with torch.amp.autocast(device_type=device.type, enabled=use_amp):
            out  = model(drug_batch=db, expr=expr, y=y)
            loss = out['loss']
        if torch.isnan(loss) or torch.isinf(loss):
            diag = _nan_diagnostics(out, y)
            raise RuntimeError(
                f'NaN/Inf loss encountered during training.\n{diag}\n'
                'Potential causes: exploding activations, too-high LR, or degenerate batch.\n'
                'Try: lower lr, higher lam_warmup_epochs, check LN_IC50 target distribution.')
        if use_amp:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            scaler.step(optimizer); scaler.update()
        else:
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()
        total_loss += loss.item() * len(y); n += len(y)
    return total_loss / n

@torch.no_grad()
def evaluate(model, loader, device, desc='eval', return_preds=False):
    model.eval()
    preds, trues, drug_ids, cosmic_ids, epistemic, aleatoric = [], [], [], [], [], []
    for batch in tqdm(loader, desc=f'  {desc}', leave=False):
        db   = batch['drug_batch'].to(device)
        expr = batch['expr'].to(device)
        # AMP disabled in eval — evidential head needs full float32 precision
        with torch.amp.autocast(device_type=device.type, enabled=False):
            out = model(drug_batch=db, expr=expr)
        preds.append(out['pred']['pred'].cpu().numpy())
        trues.append(batch['y'].numpy())
        epistemic.append(out['pred']['epistemic'].cpu().numpy())
        aleatoric.append(out['pred']['aleatoric'].cpu().numpy())
        drug_ids.append(batch['drug_ids'].numpy())
        cosmic_ids.append(batch['cosmic_ids'].numpy())
    y_pred = np.concatenate(preds); y_true = np.concatenate(trues)
    epi    = np.concatenate(epistemic); alet = np.concatenate(aleatoric)
    dids   = np.concatenate(drug_ids)
    rep = regression_report(y_true, y_pred, drug_ids=dids, uncertainties=epi)
    rep['epistemic_mean'] = float(np.nan_to_num(epi).mean())
    rep['aleatoric_mean'] = float(np.nan_to_num(alet).mean())
    if return_preds:
        rep['_preds'] = {'y_true': y_true, 'y_pred': y_pred, 'epistemic': epi,
                         'aleatoric': alet, 'drug_ids': dids,
                         'cosmic_ids': np.concatenate(cosmic_ids)}
    return rep

print('Training functions defined.')


Training functions defined.


In [21]:
# -- 5.3  Main training driver -----------------------------------------------

def train_pathxdrp(cfg, df, expr_matrix, pathway_gene_symbols, label='model'):
    set_seed(cfg['seed'])
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'\n{"="*70}'); print(f'Training {label}  (device={device})')
    print(f'Encoder: drug={cfg["drug_encoder_type"]}  cell={cfg["cell_encoder_type"]}')
    print(f'{"="*70}')

    # Build molecular graph cache
    print('Building graph cache ...')
    drugs_df = df[['DRUG_ID','SMILES']].drop_duplicates()
    graph_cache, _ = build_graph_cache(drugs_df)

    # Pathway gene map (indices)
    gene_list = list(expr_matrix.columns)
    gene_to_idx = {g: i for i, g in enumerate(gene_list)}
    pathway_gene_map = {
        pw: [gene_to_idx[g] for g in genes if g in gene_to_idx]
        for pw, genes in pathway_gene_symbols.items()
        if any(g in gene_to_idx for g in genes)
    }
    n_pairs = sum(len(v) for v in pathway_gene_map.values())
    print(f'Pathway map: {len(pathway_gene_map)} pathways | {n_pairs:,} (pw,gene) pairs')

    # Splits
    tr_idx, va_idx, te_idx = load_split(cfg['split'], cfg['seed'], cfg['fold'])
    print(f'Split sizes: train={len(tr_idx):,} val={len(va_idx):,} test={len(te_idx):,}')

    tr_ds = GDSCDataset(df.iloc[tr_idx], graph_cache, expr_matrix)
    va_ds = GDSCDataset(df.iloc[va_idx], graph_cache, expr_matrix)
    te_ds = GDSCDataset(df.iloc[te_idx], graph_cache, expr_matrix)
    _nw = 2  # Kaggle: 2 workers is safer than 4 to avoid DataLoader hangs
    lk = dict(batch_size=cfg['batch_size'], collate_fn=collate_fn,
              num_workers=_nw, pin_memory=(device.type=='cuda'), persistent_workers=(_nw>0))
    tr_l = DataLoader(tr_ds, shuffle=True,  **lk)
    va_l = DataLoader(va_ds, shuffle=False, **lk)
    te_l = DataLoader(te_ds, shuffle=False, **lk)

    # Model — use BOND_FEAT_DIM constant so edge_dim is always consistent with bond_features()
    sample_g = next(iter(graph_cache.values()))
    node_dim = sample_g.x.size(1)
    edge_dim = sample_g.edge_attr.size(1) if sample_g.edge_attr is not None else BOND_FEAT_DIM
    model = PathXDRP(
        node_in_dim=node_dim, edge_in_dim=edge_dim,
        n_genes=expr_matrix.shape[1], pathway_gene_map=pathway_gene_map,
        hidden_dim=cfg['hidden_dim'], n_gat_layers=cfg['n_gat_layers'],
        n_attn_heads=cfg['n_attn_heads'], dropout=cfg['dropout'],
        mask_type=cfg['mask_type'], evidential_lam=cfg['evidential_lam'],
        n_pw_transformer_layers=cfg['n_pw_transformer_layers'],
        drug_encoder_type=cfg['drug_encoder_type'],
        cell_encoder_type=cfg['cell_encoder_type'],
        gene_symbols=gene_list if cfg['cell_encoder_type']=='gene_mamba' else None,
        graph_mamba_kwargs=cfg.get('graph_mamba_kwargs'),
        gene_mamba_kwargs=cfg.get('gene_mamba_kwargs'),
    ).to(device)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Model parameters: {n_params:,}')

    scaler = None  # Float32 training — AMP (fp16) causes NaN with GATv2Conv+TransformerEncoder on T4
    opt = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=1e-4)
    wu = torch.optim.lr_scheduler.LinearLR(opt, start_factor=0.1, end_factor=1.0, total_iters=5)
    cos = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(cfg['epochs']-5,1))
    sched = torch.optim.lr_scheduler.SequentialLR(opt, [wu, cos], milestones=[5])

    lam_warmup = cfg.get('lam_warmup_epochs', 50)
    ckpt_dir = WORK / 'checkpoints'; ckpt_dir.mkdir(exist_ok=True)
    ckpt_path = ckpt_dir / f'{label}_{cfg["split"]}_seed{cfg["seed"]}_fold{cfg["fold"]}.pt'

    best_pcc, best_epoch = -float('inf'), 0
    epoch_pbar = tqdm(range(1, cfg['epochs']+1), desc=f'{label} epochs', unit='ep')
    for epoch in epoch_pbar:
        model.evidential_lam = cfg['evidential_lam'] * min(epoch/max(lam_warmup,1), 1.0) if lam_warmup > 0 else cfg['evidential_lam']
        tr_loss = train_one_epoch(model, tr_l, opt, device, scaler)
        va_met  = evaluate(model, va_l, device, desc='val')
        sched.step()
        if va_met['PCC'] > best_pcc:
            best_pcc = va_met['PCC']; best_epoch = epoch
            torch.save(model.state_dict(), ckpt_path)
        _pcc_str = f"{va_met['PCC']:.4f}" if isinstance(va_met['PCC'], float) and not (va_met['PCC'] != va_met['PCC']) else 'NaN'
        epoch_pbar.set_postfix(loss=f"{tr_loss:.4f}", PCC=_pcc_str,
                               best=f"{best_pcc:.4f}@{best_epoch}")

    # Test evaluation
    print('Loading best checkpoint ...')
    model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))
    te_met = evaluate(model, te_l, device, desc='test', return_preds=True)

    # Save results
    res_dir = WORK / 'results'; res_dir.mkdir(exist_ok=True)
    res_path = res_dir / f'{label}_{cfg["split"]}_seed{cfg["seed"]}_fold{cfg["fold"]}.json'
    preds_path = res_dir / f'{label}_{cfg["split"]}_seed{cfg["seed"]}_fold{cfg["fold"]}_preds.csv'
    result_clean = {k: v for k, v in te_met.items() if not k.startswith('_')}
    with open(res_path, 'w') as f: json.dump(result_clean, f, indent=2)
    p = te_met['_preds']
    pd.DataFrame({'y_true':p['y_true'],'y_pred':p['y_pred'],
                  'epistemic':p['epistemic'],'drug_id':p['drug_ids'],
                  'cosmic_id':p['cosmic_ids']}).to_csv(preds_path, index=False)

    print(f'\n=== {label} Test Results ===')
    for k, v in result_clean.items():
        if not isinstance(v, dict): print(f'  {k:25s}: {v}')
    print(f'Saved: {res_path}')
    return result_clean

print('train_pathxdrp() defined.')


train_pathxdrp() defined.


---
## 6. Version A — No Mamba (GAT + PathwaySet)

In [22]:
CFG_A = dict(
    split            = 'random',   # random | cell_blind | drug_blind | tissue_blind
    seed             = 0,
    fold             = 0,
    drug_encoder_type = 'gat',
    cell_encoder_type = 'pathway_set',
    hidden_dim       = 256,
    n_gat_layers     = 4,
    n_attn_heads     = 8,
    dropout          = 0.1,
    mask_type        = 'soft',
    n_pw_transformer_layers = 1,
    evidential_lam   = 0.01,
    lam_warmup_epochs = 50,
    batch_size       = 256,
    epochs           = 150,
    lr               = 1e-3,
)

results_a = train_pathxdrp(CFG_A, df, expr_matrix, pathway_gene_symbols, label='no_mamba')


Training no_mamba  (device=cuda)
Encoder: drug=gat  cell=pathway_set
Building graph cache ...
  Building FG vocab from 247 SMILES ...


  Drug graphs:   0%|          | 0/247 [00:00<?, ?it/s]

  Graph cache: 247 drugs (0 failed)
Pathway map: 370 pathways | 37,753 (pw,gene) pairs
Split sizes: train=120,940 val=15,118 test=15,118
Model parameters: 1,815,414


no_mamba epochs:   0%|          | 0/150 [00:00<?, ?ep/s]

  train:   0%|          | 0/473 [00:00<?, ?it/s]

...

  train:   0%|          | 0/473 [00:00<?, ?it/s]

  val:   0%|          | 0/60 [00:00<?, ?it/s]

Loading best checkpoint ...


  test:   0%|          | 0/60 [00:00<?, ?it/s]


=== no_mamba Test Results ===
  RMSE                     : 1.0151127623886815
  MAE                      : 0.7413613200187683
  PCC                      : 0.9355348348617554
  Spearman                 : 0.9065573992301647
  R2                       : 0.8717445135116577
  Per-drug PCC             : 0.7423330730030894
  ECE                      : 0.38084420561790466
  epistemic_mean           : 0.6055602431297302
  aleatoric_mean           : 0.07923674583435059
Saved: /kaggle/working/results/no_mamba_random_seed0_fold0.json


---
## 7. Version B — With Mamba (GraphMamba + GeneMamba)

In [23]:
# Pre-download GeneMamba backbone weights so training starts cleanly
print('Pre-downloading GeneMamba backbone (~250 MB) ...')
from transformers import AutoModel
_bk = AutoModel.from_pretrained('mineself2016/GeneMamba', trust_remote_code=True)
print(f'GeneMamba: {sum(p.numel() for p in _bk.parameters()):,} params')
del _bk; import gc; gc.collect(); torch.cuda.empty_cache()

CFG_B = dict(
    split            = 'random',
    seed             = 0,
    fold             = 0,
    drug_encoder_type = 'graph_mamba',
    cell_encoder_type = 'gene_mamba',
    hidden_dim       = 256,
    n_gat_layers     = 4,   # unused directly (graph_mamba_kwargs used instead)
    n_attn_heads     = 8,
    dropout          = 0.1,
    mask_type        = 'soft',
    n_pw_transformer_layers = 1,
    evidential_lam   = 0.01,
    lam_warmup_epochs = 50,
    batch_size       = 128,   # smaller: backbone uses extra VRAM
    epochs           = 150,
    lr               = 5e-4,
    graph_mamba_kwargs = {'n_gat_layers': 2, 'n_mamba_layers': 2, 'ordering': 'degree'},
    gene_mamba_kwargs  = {'top_k': 2048, 'backbone_id': 'mineself2016/GeneMamba',
                          'freeze_backbone': True},
)

results_b = train_pathxdrp(CFG_B, df, expr_matrix, pathway_gene_symbols, label='with_mamba')


Pre-downloading GeneMamba backbone (~250 MB) ...


config.json:   0%|          | 0.00/800 [00:00<?, ?B/s]

configuration_genemamba.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/mineself2016/GeneMamba:
- configuration_genemamba.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_genemamba.py: 0.00B [00:00, ?B/s]

modeling_outputs.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/mineself2016/GeneMamba:
- modeling_outputs.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/mineself2016/GeneMamba:
- modeling_genemamba.py
- modeling_outputs.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/263M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/194 [00:00<?, ?it/s]

AttributeError: 'GeneMambaModel' object has no attribute 'all_tied_weights_keys'

---
## 8. Results Comparison & Export

In [ ]:
import zipfile, datetime

rows = []
for label, res in [('No Mamba (GAT+PathwaySet)', results_a),
                   ('With Mamba (GraphMamba+GeneMamba)', results_b)]:
    if res is None: continue
    row = {'Model': label}
    for k in ['PCC','RMSE','R2','Spearman','MAE','ECE','epistemic_mean']:
        row[k] = round(res.get(k, float('nan')), 4)
    rows.append(row)

if rows:
    cmp_df = pd.DataFrame(rows).set_index('Model')
    print('\n=== Final Comparison ===')
    print(cmp_df.to_string())
    cmp_df.to_csv(WORK / 'results' / 'comparison.csv')

# Zip everything for download
ts = datetime.datetime.now().strftime('%Y%m%d_%H%M')
zip_path = WORK / f'pathxdrp_outputs_{ts}.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for pt in (WORK/'checkpoints').glob('*.pt'):
        zf.write(pt, f'checkpoints/{pt.name}')
    for jf in (WORK/'results').glob('*.json'):
        zf.write(jf, f'results/{jf.name}')
    for cf in (WORK/'results').glob('*.csv'):
        zf.write(cf, f'results/{cf.name}')
print(f'\nDownloadable zip: {zip_path}  ({zip_path.stat().st_size/1024**2:.1f} MB)')
print('Go to Kaggle Output tab to download.')